<a href="https://colab.research.google.com/github/rayeeed/UFP_Project/blob/main/PM2_5_models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ***Wet Model***

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import IsolationForest
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import LinearRegression
from xgboost import XGBRegressor
#from sklearn.model_selection import LeaveOneOut
from sklearn.model_selection import cross_val_score,cross_val_predict
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score
from sklearn.metrics import r2_score
from sklearn.metrics import mean_absolute_error
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import RidgeCV
from sklearn.feature_selection import mutual_info_regression
from sklearn.feature_selection import SelectFromModel
from time import time
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import RepeatedKFold
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error
import seaborn as sns
from sklearn.preprocessing import Normalizer,MinMaxScaler,StandardScaler,RobustScaler
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge, Lasso, BayesianRidge
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.feature_selection import RFE,RFECV
from sklearn.linear_model import (HuberRegressor,
                              	RANSACRegressor, TheilSenRegressor)

In [ ]:
!pip install shap
import shap

In [ ]:
!pip install optuna
import optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 404.7/404.7 kB 23.9 MB/s eta 0:00:00


In [ ]:
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.preprocessing import KBinsDiscretizer

# Load the dataset
df = pd.read_excel('/content/drive/MyDrive/datasets_Rayeed_GSV_POI_LU/model_35_wet_dry.xlsx', index_col='site_name')

# Drop unnecessary columns
df.drop(['Site_type', 'OID'], inplace=True, axis=1)

# Separate the target variable and features
y = df['Mean_wet_pm']
X = df.drop(['Mean PNC (# / cm3)', 'Mean PM2.5 (µg/m3)','Mean_wet_pm','Mean_dry_pm','Mean_wet_pnc','Mean_dry_pnc'], axis=1)

In [ ]:
# Discretize the target variable into bins using 'quantile' strategy
binner = KBinsDiscretizer(n_bins=5, encode='ordinal', strategy='quantile')
y_binned = binner.fit_transform(y.values.reshape(-1, 1)).flatten()

# Perform stratified train-test split
split = StratifiedShuffleSplit(n_splits=1, test_size=.2, random_state=42)
for train_index, test_index in split.split(X, y_binned):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

# Check the distribution of bins in the train and test sets
print("Train bin distribution:\n", pd.value_counts(y_binned[train_index], sort=False))
print("Test bin distribution:\n", pd.value_counts(y_binned[test_index], sort=False))

# Output the resulting dataframes
print("X_train:\n", X_train.head())
print("y_train:\n", y_train.head())
print("X_test:\n", X_test.head())
print("y_test:\n", y_test.head())

Train bin distribution:
 0.0    5
1.0    6
3.0    5
4.0    6
2.0    6
Name: count, dtype: int64
Test bin distribution:
 0.0    2
1.0    1
3.0    2
2.0    1
4.0    1
Name: count, dtype: int64
X_train:
                  gsv_wall_p100  gsv_building_p100  gsv_house_p100  \
site_name                                                           
Shangsad_bhaban        0.10000           2.592500        0.000000   
Azimpur                0.00000           0.000000        0.000000   
Uttara_Sector7         6.76536          32.859999        0.233735   
Aftabnagar             3.69000          32.427500        0.257500   
Shimanto_square        0.31500          15.397500        0.000000   

                 gsv_awning_p100  gsv_sky_p100  gsv_earth_p100  gsv_tree_p100  \
site_name                                                                       
Shangsad_bhaban         0.000000     28.177500        3.552500      36.085000   
Azimpur                 0.000000      0.000000        0.000000       0.0

/tmp/ipython-input-3543545133.py:12: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  print("Train bin distribution:\n", pd.value_counts(y_binned[train_index], sort=False))
/tmp/ipython-input-3543545133.py:13: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  print("Test bin distribution:\n", pd.value_counts(y_binned[test_index], sort=False))


In [ ]:
from scipy.stats import ks_2samp
ks_statistic, p_value = ks_2samp(y_train, y_test)

print(f"K-S statistic: {ks_statistic}")
print(f"P-value: {p_value}")

# Interpretation
alpha = 0.05
if p_value < alpha:
    print("The null hypothesis is rejected. The distributions of the train and test sets are different.")
else:
    print("The null hypothesis cannot be rejected. The distributions of the train and test sets are the same.")

K-S statistic: 0.17857142857142858
P-value: 0.9883820703931286
The null hypothesis cannot be rejected. The distributions of the train and test sets are the same.


In [ ]:
# Apply Kolmogorov-Smirnov test for each feature
ks_results = {}
failed_features = []
alpha = 0.05
for feature in X.columns:
    ks_statistic, p_value = ks_2samp(X_train[feature], X_test[feature])
    ks_results[feature] = {'ks_statistic': ks_statistic, 'p_value': p_value}
    if p_value < alpha:
        failed_features.append(feature)

# Print K-S test results for each feature
for feature, result in ks_results.items():
    print(f"Feature: {feature}")
    print(f"K-S statistic: {result['ks_statistic']}")
    print(f"P-value: {result['p_value']}")
    if result['p_value'] < alpha:
        print("The null hypothesis is rejected. The distributions of the train and test sets are different for this feature.\n")
    else:
        print("The null hypothesis cannot be rejected. The distributions of the train and test sets are the same for this feature.\n")

# Print features that failed the K-S test
if failed_features:
    print("Features that failed the K-S test (null hypothesis rejected):")
    for feature in failed_features:
        print(f"- {feature}")
else:
    print("All features passed the K-S test (null hypothesis not rejected).")

Feature: gsv_wall_p100
K-S statistic: 0.32142857142857145
P-value: 0.5556032252116138
The null hypothesis cannot be rejected. The distributions of the train and test sets are the same for this feature.

Feature: gsv_building_p100
K-S statistic: 0.5357142857142857
P-value: 0.06189438056545301
The null hypothesis cannot be rejected. The distributions of the train and test sets are the same for this feature.

Feature: gsv_house_p100
K-S statistic: 0.2857142857142857
P-value: 0.7050303962215889
The null hypothesis cannot be rejected. The distributions of the train and test sets are the same for this feature.

Feature: gsv_awning_p100
K-S statistic: 0.2857142857142857
P-value: 0.7050303962215889
The null hypothesis cannot be rejected. The distributions of the train and test sets are the same for this feature.

Feature: gsv_sky_p100
K-S statistic: 0.2857142857142857
P-value: 0.7050303962215889
The null hypothesis cannot be rejected. The distributions of the train and test sets are the same f

In [ ]:
X_train_dropped = X_train.drop(columns=failed_features)
X_test_dropped = X_test.drop(columns=failed_features)

In [ ]:
trans = StandardScaler()
X_st = trans.fit_transform(X_train_dropped)
X_st = pd.DataFrame(X_st, columns=X_train_dropped.columns, index=X_train_dropped.index)

In [ ]:
X_test_st = trans.transform(X_test_dropped)
X_test_st = pd.DataFrame(X_test_st, columns=X_test_dropped.columns, index=X_test_dropped.index)

In [ ]:
X_Pred = pd.read_excel('/content/drive/MyDrive/datasets_Rayeed_GSV_POI_LU/combined_total_v4.0.xlsx', index_col='OID_')

In [ ]:
col=X_st.columns
X_Pred_f1 = X_Pred[col]
X_Pred_f2= X_Pred_f1

In [ ]:
x_pred_st= trans.transform(X_Pred_f2)
x_pred_st = pd.DataFrame(x_pred_st, columns = X_Pred_f2.columns,index=X_Pred_f2.index)

SVR

In [ ]:
import pandas as pd
import optuna
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import cross_val_predict

# Feature set to be used for prediction
features = ['distPrim', 'gsv_fence_p1500', 'gsv_earth_p1500']

def evaluate_model(features):
    X_Train = X_st[features]
    X_test = X_test_st[features]

    def objective(trial):
        # 1. Fixed Kernel
        kernel = 'linear'

        # 2. Optimize Hyperparameters for Linear Kernel
        C = trial.suggest_float("C", 0.01, 1000000.0, log=True)
        epsilon = trial.suggest_float("epsilon", 0.001, 10.0)

        # 3. Train and Evaluate
        svr = SVR(kernel=kernel, C=C, epsilon=epsilon)

        # Optuna minimizes the returned value (MSE)
        predicted = cross_val_predict(svr, X_Train, y_train, cv=10)
        return mean_squared_error(y_train, predicted)

    # Create and run study
    study = optuna.create_study(direction='minimize', study_name="SVR_Linear_RMSE")
    study.optimize(objective, n_trials=100) # Reduced trials since search space is smaller

    # --- Rebuild Final Model ---
    best_params = study.best_params

    final_C = best_params['C']
    final_epsilon = best_params['epsilon']

    model = SVR(
        kernel='linear',
        C=final_C,
        epsilon=final_epsilon
    )

    model.fit(X_Train, y_train)

    # --- Calculate True RMSE (Square root of MSE) ---

    # 1. Training Performance
    pred_train = model.predict(X_Train)
    rmse_train = mean_squared_error(y_train, pred_train) ** 0.5
    r2_train = r2_score(y_train, pred_train)

    # 2. Cross-Validation Performance
    predicted_cv = cross_val_predict(model, X_Train, y_train, cv=10)
    rmse_cv = mean_squared_error(y_train, predicted_cv) ** 0.5
    r2_cv = r2_score(y_train, predicted_cv)

    # 3. Test Performance
    predicted_test = model.predict(X_test)
    rmse_test = mean_squared_error(y_test, predicted_test) ** 0.5
    r2_test = r2_score(y_test, predicted_test)

    return {
        "features": features,
        "best_params": best_params,
        "model": model,
        "rmse_train": rmse_train,
        "r2_train": r2_train,
        "rmse_cv": rmse_cv,
        "r2_cv": r2_cv,
        "rmse_test": rmse_test,
        "r2_test": r2_test
    }

print(f"Evaluating feature set (Linear Kernel Only): {features}")

# --- Execution Block ---
try:
    result = evaluate_model(features)
    results = [result]

    print("Best Parameters Found:", result["best_params"])
    print(f"Test RMSE: {result['rmse_test']:.4f}")

except ValueError as e:
    if "Input contains NaN" in str(e):
        print(f"Skipping evaluation due to NaN values in feature set: {features}")
        results = []
    else:
        raise e

# --- New Prediction & Saving Logic ---
if results:
    print("\n--- Generating Predictions for New Dataset ---")

    # Get the trained model
    model = results[0]['model']

    # Generate predictions for the new dataset
    x_pred_st_subset = x_pred_st[features]
    predictions = model.predict(x_pred_st_subset)

    # 1. Save Predictions
    predictions_df = pd.DataFrame(predictions, index=x_pred_st.index, columns=["Predictions"])
    predictions_df.to_csv("/content/drive/MyDrive/datasets_Rayeed_GSV_POI_LU/predictions_wet_pm25_svr.csv")

    # 2. Save Prediction Stats
    stats = predictions_df.describe()
    stats.to_csv("/content/drive/MyDrive/datasets_Rayeed_GSV_POI_LU/predictions_wet_pm25_svr_stats.csv")

    # 3. Save Model Performance Metrics (RMSE, R2)
    metrics_data = {
        "RMSE_Train": results[0]['rmse_train'],
        "R2_Train": results[0]['r2_train'],
        "RMSE_CV": results[0]['rmse_cv'],
        "R2_CV": results[0]['r2_cv'],
        "RMSE_Test": results[0]['rmse_test'],
        "R2_Test": results[0]['r2_test']
    }

    metrics_df = pd.DataFrame([metrics_data])
    metrics_df.to_csv("/content/drive/MyDrive/datasets_Rayeed_GSV_POI_LU/svr_wt_pm_model_performance_metrics.csv", index=False)

    print("\nHead of Predictions:")
    print(predictions_df.head())

    print("\nPrediction Statistics:")
    print(stats)

    print("\nModel Performance Metrics:")
    print(metrics_df)

    print("\nFinal Result Dictionary:")
    print(results)
else:
    print("No model was trained; skipping prediction generation.")

[I 2025-12-21 20:32:27,419] A new study created in memory with name: SVR_Linear_RMSE
[I 2025-12-21 20:32:27,504] Trial 0 finished with value: 109.90484359015848 and parameters: {'C': 4.606684881898956, 'epsilon': 6.542767581032391}. Best is trial 0 with value: 109.90484359015848.
[I 2025-12-21 20:32:27,586] Trial 1 finished with value: 142.33940122591588 and parameters: {'C': 0.0780219375520174, 'epsilon': 7.585150187782696}. Best is trial 0 with value: 109.90484359015848.


Evaluating feature set (Linear Kernel Only): ['distPrim', 'gsv_fence_p1500', 'gsv_earth_p1500']


[I 2025-12-21 20:32:27,742] Trial 2 finished with value: 140.63222457759153 and parameters: {'C': 0.11196648117036753, 'epsilon': 7.577593507374975}. Best is trial 0 with value: 109.90484359015848.
[I 2025-12-21 20:32:27,876] Trial 3 finished with value: 121.23299781353894 and parameters: {'C': 584.9059590482317, 'epsilon': 7.666840069854784}. Best is trial 0 with value: 109.90484359015848.
[I 2025-12-21 20:32:28,655] Trial 4 finished with value: 99.62194609210975 and parameters: {'C': 5419.5541445991175, 'epsilon': 5.512846256323088}. Best is trial 4 with value: 99.62194609210975.
[I 2025-12-21 20:32:52,736] Trial 5 finished with value: 158.92936210199315 and parameters: {'C': 190500.70508293647, 'epsilon': 0.8515123217833752}. Best is trial 4 with value: 99.62194609210975.
[I 2025-12-21 20:32:53,025] Trial 6 finished with value: 125.04958129149452 and parameters: {'C': 19516.7374257448, 'epsilon': 7.80417704055763}. Best is trial 4 with value: 99.62194609210975.
[I 2025-12-21 20:32:5

Best Parameters Found: {'C': 28.912428658813557, 'epsilon': 6.279168849823631}
Test RMSE: 6.6127

--- Generating Predictions for New Dataset ---

Head of Predictions:
      Predictions
OID_             
1268    58.926407
1271    55.045864
1272    52.800538
1273    43.308970
1274    44.069237

Prediction Statistics:
       Predictions
count  3724.000000
mean     59.786508
std       8.344368
min      35.323295
25%      54.955262
50%      59.537080
75%      64.306135
max      90.460741

Model Performance Metrics:
   RMSE_Train  R2_Train   RMSE_CV     R2_CV  RMSE_Test   R2_Test
0    8.946259  0.406524  9.398157  0.345054   6.612685  0.251501

Final Result Dictionary:
[{'features': ['distPrim', 'gsv_fence_p1500', 'gsv_earth_p1500'], 'best_params': {'C': 28.912428658813557, 'epsilon': 6.279168849823631}, 'model': SVR(C=28.912428658813557, epsilon=6.279168849823631, kernel='linear'), 'rmse_train': 8.946258799773583, 'r2_train': 0.40652378060337113, 'rmse_cv': 9.398157192539344, 'r2_cv': 0.345

LR

In [ ]:
import pandas as pd
import optuna
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import cross_val_predict

# Feature set to be used for prediction
features = ['distPrim', 'gsv_fence_p1500', 'gsv_earth_p1500']

def evaluate_model(features):
    X_Train = X_st[features]
    X_test = X_test_st[features]

    def objective(trial):
        # No hyperparameters to optimize for LinearRegression
        model = LinearRegression()
        predicted = cross_val_predict(model, X_Train, y_train, cv=10)
        # Calculate RMSE: MSE ** 0.5
        return mean_squared_error(y_train, predicted) ** 0.5

    # Run study (Structure kept for consistency, even though LR has no params)
    study = optuna.create_study(direction='minimize', study_name="LinearRegression_RMSE")
    study.optimize(objective, n_trials=1)

    model = LinearRegression()
    model.fit(X_Train, y_train)

    # --- Calculate True RMSE (Square root of MSE) ---

    # 1. Training Performance
    pred_train = model.predict(X_Train)
    rmse_train = mean_squared_error(y_train, pred_train) ** 0.5
    r2_train = r2_score(y_train, pred_train)

    # 2. Cross-Validation Performance
    predicted_cv = cross_val_predict(model, X_Train, y_train, cv=10)
    rmse_cv = mean_squared_error(y_train, predicted_cv) ** 0.5
    r2_cv = r2_score(y_train, predicted_cv)

    # 3. Test Performance
    predicted_test = model.predict(X_test)
    rmse_test = mean_squared_error(y_test, predicted_test) ** 0.5
    r2_test = r2_score(y_test, predicted_test)

    return {
        "features": features,
        "model": model,
        "rmse_train": rmse_train,
        "r2_train": r2_train,
        "rmse_cv": rmse_cv,
        "r2_cv": r2_cv,
        "rmse_test": rmse_test,
        "r2_test": r2_test
    }

# Evaluate the model using the specified feature set
print(f"Evaluating feature set: {features}")

try:
    result = evaluate_model(features)
    results = [result]
    print(f"Test RMSE: {result['rmse_test']:.4f}")

except ValueError as e:
    if "Input contains NaN" in str(e):
        print(f"Skipping evaluation due to NaN values in feature set: {features}")
        results = []
    else:
        raise e

# --- New Prediction & Saving Logic ---
if results:
    print("\n--- Generating Predictions for New Dataset ---")

    # Get the trained model
    model = results[0]['model']

    # Generate predictions for the new dataset
    x_pred_st_subset = x_pred_st[features]
    predictions = model.predict(x_pred_st_subset)

    # 1. Save Predictions
    predictions_df = pd.DataFrame(predictions, index=x_pred_st.index, columns=["Predictions"])
    predictions_df.to_csv("/content/drive/MyDrive/datasets_Rayeed_GSV_POI_LU/predictions_wet_pm25_linear_regression.csv")

    # 2. Save Prediction Stats
    stats = predictions_df.describe()
    stats.to_csv("/content/drive/MyDrive/datasets_Rayeed_GSV_POI_LU/predictions_wet_pm25_linear_regression_stats.csv")

    # 3. Save Model Performance Metrics (RMSE, R2)
    metrics_data = {
        "RMSE_Train": results[0]['rmse_train'],
        "R2_Train": results[0]['r2_train'],
        "RMSE_CV": results[0]['rmse_cv'],
        "R2_CV": results[0]['r2_cv'],
        "RMSE_Test": results[0]['rmse_test'],
        "R2_Test": results[0]['r2_test']
    }

    metrics_df = pd.DataFrame([metrics_data])
    metrics_df.to_csv("/content/drive/MyDrive/datasets_Rayeed_GSV_POI_LU/linear_regression_wet_pm_model_performance_metrics.csv", index=False)

    print("\nHead of Predictions:")
    print(predictions_df.head())

    print("\nPrediction Statistics:")
    print(stats)

    print("\nModel Performance Metrics:")
    print(metrics_df)

    print("\nFinal Result Dictionary:")
    print(results)
else:
    print("No model was trained; skipping prediction generation.")

[I 2025-12-21 20:35:34,668] A new study created in memory with name: LinearRegression_RMSE
[I 2025-12-21 20:35:34,735] Trial 0 finished with value: 11.442761614843453 and parameters: {}. Best is trial 0 with value: 11.442761614843453.


Evaluating feature set: ['distPrim', 'gsv_fence_p1500', 'gsv_earth_p1500']
Test RMSE: 6.7066

--- Generating Predictions for New Dataset ---

Head of Predictions:
      Predictions
OID_             
1268    57.571915
1271    53.738953
1272    51.331403
1273    33.407318
1274    34.858051

Prediction Statistics:
       Predictions
count  3724.000000
mean     57.843930
std       8.038690
min      31.506642
25%      53.159400
50%      57.286632
75%      61.762838
max      89.190972

Model Performance Metrics:
   RMSE_Train  R2_Train    RMSE_CV     R2_CV  RMSE_Test   R2_Test
0    8.763788  0.430486  11.442762  0.029083    6.70658  0.230094

Final Result Dictionary:
[{'features': ['distPrim', 'gsv_fence_p1500', 'gsv_earth_p1500'], 'model': LinearRegression(), 'rmse_train': 8.76378835733906, 'r2_train': 0.43048630948439415, 'rmse_cv': 11.442761614843453, 'r2_cv': 0.029082995023503533, 'rmse_test': 6.706579522740388, 'r2_test': 0.23009388437247968}]


XGBoost

In [ ]:
import pandas as pd
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import cross_val_predict
from xgboost import XGBRegressor
import optuna

# Feature set to be used for prediction
features = ['distPrim', 'gsv_fence_p1500', 'gsv_earth_p1500']


# Function to run the optimization and evaluate the model
def evaluate_model(features):
    X_Train = X_st[features]
    X_test = X_test_st[features]

    def objective(trial):
        # Optimization parameters
        learning_rate = trial.suggest_float('learning_rate', 0.005, 1.5)
        n_estimators = trial.suggest_int('n_estimators', 50, 1000)
        reg_alpha = trial.suggest_float('reg_alpha', 0.001, 100)
        reg_lambda = trial.suggest_float('reg_lambda', 0.001, 3)

        # Note: You can uncomment depth/weight if switching to 'gbtree' booster later
        # max_depth = trial.suggest_int('max_depth', 2, 10)

        XGB = XGBRegressor(
            booster='gblinear', # keeping linear booster as per your snippet
            n_estimators=n_estimators,
            learning_rate=learning_rate,
            reg_alpha=reg_alpha,
            reg_lambda=reg_lambda,
            objective="reg:squarederror",
            seed=42,
            verbosity=0 # Silence xgboost warnings during optimization
        )

        predicted = cross_val_predict(XGB, X_Train, y_train, cv=10)
        return mean_squared_error(y_train, predicted) ** 0.5

    study = optuna.create_study(direction='minimize', study_name="XGBRegressor_RMSE")
    study.optimize(objective, n_trials=500)

    best_params = study.best_params

    # Rebuild final model with best params
    model = XGBRegressor(
        booster='gblinear',
        n_estimators=best_params['n_estimators'],
        learning_rate=best_params['learning_rate'],
        reg_alpha=best_params['reg_alpha'],
        reg_lambda=best_params['reg_lambda'],
        objective="reg:squarederror",
        seed=42
    )

    model.fit(X_Train, y_train)

    # --- Calculate True RMSE (Square root of MSE) ---

    # 1. Training Performance
    pred_train = model.predict(X_Train)
    rmse_train = mean_squared_error(y_train, pred_train) ** 0.5
    r2_train = r2_score(y_train, pred_train)

    # 2. Cross-Validation Performance
    predicted_cv = cross_val_predict(model, X_Train, y_train, cv=10)
    rmse_cv = mean_squared_error(y_train, predicted_cv) ** 0.5
    r2_cv = r2_score(y_train, predicted_cv)

    # 3. Test Performance
    predicted_test = model.predict(X_test)
    rmse_test = mean_squared_error(y_test, predicted_test) ** 0.5
    r2_test = r2_score(y_test, predicted_test)

    return {
        "features": features,
        "best_params": best_params,
        "model": model,
        "rmse_train": rmse_train,
        "r2_train": r2_train,
        "rmse_cv": rmse_cv,
        "r2_cv": r2_cv,
        "rmse_test": rmse_test,
        "r2_test": r2_test
    }

# Evaluate the model using the specified feature set
print(f"Evaluating feature set: {features}")

try:
    result = evaluate_model(features)
    results = [result]
    print("Best Parameters Found:", result["best_params"])
    print(f"Test RMSE: {result['rmse_test']:.4f}")

except ValueError as e:
    if "Input contains NaN" in str(e):
        print(f"Skipping evaluation due to NaN values in feature set: {features}")
        results = []
    else:
        raise e

# --- New Prediction & Saving Logic ---
if results:
    print("\n--- Generating Predictions for New Dataset ---")

    # Get the trained model
    model = results[0]['model']

    # Generate predictions for the new dataset
    x_pred_st_subset = x_pred_st[features]
    predictions = model.predict(x_pred_st_subset)

    # 1. Save Predictions
    predictions_df = pd.DataFrame(predictions, index=x_pred_st.index, columns=["Predictions"])
    predictions_df.to_csv("/content/drive/MyDrive/datasets_Rayeed_GSV_POI_LU/predictions_wet_pm25_xgb.csv")

    # 2. Save Prediction Stats
    stats = predictions_df.describe()
    stats.to_csv("/content/drive/MyDrive/datasets_Rayeed_GSV_POI_LU/predictions_wet_pm25_xgb_stats.csv")

    # 3. Save Model Performance Metrics (RMSE, R2)
    metrics_data = {
        "RMSE_Train": results[0]['rmse_train'],
        "R2_Train": results[0]['r2_train'],
        "RMSE_CV": results[0]['rmse_cv'],
        "R2_CV": results[0]['r2_cv'],
        "RMSE_Test": results[0]['rmse_test'],
        "R2_Test": results[0]['r2_test']
    }

    metrics_df = pd.DataFrame([metrics_data])
    metrics_df.to_csv("/content/drive/MyDrive/datasets_Rayeed_GSV_POI_LU/xgb_wet_pm_model_performance_metrics.csv", index=False)

    print("\nHead of Predictions:")
    print(predictions_df.head())

    print("\nPrediction Statistics:")
    print(stats)

    print("\nModel Performance Metrics:")
    print(metrics_df)

    print("\nFinal Result Dictionary:")
    print(results)
else:
    print("No model was trained; skipping prediction generation.")

[I 2025-12-22 21:31:28,917] A new study created in memory with name: XGBRegressor_RMSE
[I 2025-12-22 21:31:29,038] Trial 0 finished with value: 12.43008093955885 and parameters: {'learning_rate': 0.7019623082308115, 'n_estimators': 57, 'reg_alpha': 10.01859090589998, 'reg_lambda': 2.0359161782424464}. Best is trial 0 with value: 12.43008093955885.


Evaluating feature set: ['distPrim', 'gsv_fence_p1500', 'gsv_earth_p1500']


[I 2025-12-22 21:31:29,655] Trial 1 finished with value: 12.43008106948076 and parameters: {'learning_rate': 0.6559494546976609, 'n_estimators': 488, 'reg_alpha': 35.4619580221033, 'reg_lambda': 1.9379448868831077}. Best is trial 0 with value: 12.43008093955885.
[I 2025-12-22 21:31:30,007] Trial 2 finished with value: 12.430081276108147 and parameters: {'learning_rate': 1.315999683930156, 'n_estimators': 516, 'reg_alpha': 10.216458008747313, 'reg_lambda': 2.457469222008513}. Best is trial 0 with value: 12.43008093955885.
[I 2025-12-22 21:31:30,364] Trial 3 finished with value: 12.43008092706656 and parameters: {'learning_rate': 1.4271377488168655, 'n_estimators': 536, 'reg_alpha': 45.52132274821321, 'reg_lambda': 1.0533982446885541}. Best is trial 3 with value: 12.43008092706656.
[I 2025-12-22 21:31:30,494] Trial 4 finished with value: 12.430080677512015 and parameters: {'learning_rate': 0.303797208026124, 'n_estimators': 107, 'reg_alpha': 17.722566664376306, 'reg_lambda': 2.3884252121

Best Parameters Found: {'learning_rate': 0.9165992627175963, 'n_estimators': 892, 'reg_alpha': 0.006860881978399456, 'reg_lambda': 0.6976165997867254}
Test RMSE: 6.7481

--- Generating Predictions for New Dataset ---

Head of Predictions:
      Predictions
OID_             
1268    57.334698
1271    54.564922
1272    52.896049
1273    39.521503
1274    40.550259

Prediction Statistics:
       Predictions
count  3724.000000
mean     57.275997
std       5.297165
min      38.308887
25%      54.143569
50%      57.086794
75%      60.012751
max      76.358383

Model Performance Metrics:
   RMSE_Train  R2_Train    RMSE_CV     R2_CV  RMSE_Test   R2_Test
0    9.088922  0.387445  10.847599  0.127455   6.748139  0.220522

Final Result Dictionary:
[{'features': ['distPrim', 'gsv_fence_p1500', 'gsv_earth_p1500'], 'best_params': {'learning_rate': 0.9165992627175963, 'n_estimators': 892, 'reg_alpha': 0.006860881978399456, 'reg_lambda': 0.6976165997867254}, 'model': XGBRegressor(base_score=None, boost

# **Ridge Regression**

In [ ]:
import pandas as pd
import optuna
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import cross_val_predict

# Feature set to be used for prediction
features = ['distPrim', 'gsv_fence_p1500', 'gsv_earth_p1500']
# Function to run the optimization and evaluate the model
def evaluate_model(features):
    X_Train = X_st[features]
    X_test = X_test_st[features]

    def objective(trial):
        # Optimization parameter for Ridge: Alpha (Regularization strength)
        # We usually search alpha on a log scale
        alpha = trial.suggest_float('alpha', 0.01, 10000.0, log=True)

        # Instantiate Ridge model
        model = Ridge(alpha=alpha, random_state=42)

        predicted = cross_val_predict(model, X_Train, y_train, cv=10)
        # Minimize RMSE
        return mean_squared_error(y_train, predicted) ** 0.5

    study = optuna.create_study(direction='minimize', study_name="Ridge_RMSE")
    study.optimize(objective, n_trials=100) # 100 trials is usually sufficient for single-parameter tuning

    best_params = study.best_params

    # Rebuild final model with best params
    model = Ridge(
        alpha=best_params['alpha'],
        random_state=42
    )

    model.fit(X_Train, y_train)

    # --- Calculate True RMSE (Square root of MSE) ---

    # 1. Training Performance
    pred_train = model.predict(X_Train)
    rmse_train = mean_squared_error(y_train, pred_train) ** 0.5
    r2_train = r2_score(y_train, pred_train)

    # 2. Cross-Validation Performance
    predicted_cv = cross_val_predict(model, X_Train, y_train, cv=10)
    rmse_cv = mean_squared_error(y_train, predicted_cv) ** 0.5
    r2_cv = r2_score(y_train, predicted_cv)

    # 3. Test Performance
    predicted_test = model.predict(X_test)
    rmse_test = mean_squared_error(y_test, predicted_test) ** 0.5
    r2_test = r2_score(y_test, predicted_test)

    return {
        "features": features,
        "best_params": best_params,
        "model": model,
        "rmse_train": rmse_train,
        "r2_train": r2_train,
        "rmse_cv": rmse_cv,
        "r2_cv": r2_cv,
        "rmse_test": rmse_test,
        "r2_test": r2_test
    }

# Evaluate the model using the specified feature set
print(f"Evaluating feature set (Ridge Regression): {features}")

try:
    result = evaluate_model(features)
    results = [result]
    print("Best Parameters Found:", result["best_params"])
    print(f"Test RMSE: {result['rmse_test']:.4f}")

except ValueError as e:
    if "Input contains NaN" in str(e):
        print(f"Skipping evaluation due to NaN values in feature set: {features}")
        results = []
    else:
        raise e

# --- New Prediction & Saving Logic ---
if results:
    print("\n--- Generating Predictions for New Dataset ---")

    # Get the trained model
    model = results[0]['model']

    # Generate predictions for the new dataset
    x_pred_st_subset = x_pred_st[features]
    predictions = model.predict(x_pred_st_subset)

    # 1. Save Predictions
    predictions_df = pd.DataFrame(predictions, index=x_pred_st.index, columns=["Predictions"])
    predictions_df.to_csv("/content/drive/MyDrive/datasets_Rayeed_GSV_POI_LU/predictions_wet_pm25_ridge.csv")

    # 2. Save Prediction Stats
    stats = predictions_df.describe()
    stats.to_csv("/content/drive/MyDrive/datasets_Rayeed_GSV_POI_LU/predictions_wet_pm25_ridge_stats.csv")

    # 3. Save Model Performance Metrics (RMSE, R2)
    metrics_data = {
        "RMSE_Train": results[0]['rmse_train'],
        "R2_Train": results[0]['r2_train'],
        "RMSE_CV": results[0]['rmse_cv'],
        "R2_CV": results[0]['r2_cv'],
        "RMSE_Test": results[0]['rmse_test'],
        "R2_Test": results[0]['r2_test']
    }

    metrics_df = pd.DataFrame([metrics_data])
    metrics_df.to_csv("/content/drive/MyDrive/datasets_Rayeed_GSV_POI_LU/ridge_wet_model_performance_metrics.csv", index=False)

    print("\nHead of Predictions:")
    print(predictions_df.head())

    print("\nPrediction Statistics:")
    print(stats)

    print("\nModel Performance Metrics:")
    print(metrics_df)

    print("\nFinal Result Dictionary:")
    print(results)
else:
    print("No model was trained; skipping prediction generation.")

[I 2025-12-22 21:53:01,027] A new study created in memory with name: Ridge_RMSE
[I 2025-12-22 21:53:01,068] Trial 0 finished with value: 12.365483816609983 and parameters: {'alpha': 2223.887589445979}. Best is trial 0 with value: 12.365483816609983.
[I 2025-12-22 21:53:01,110] Trial 1 finished with value: 11.012865424739477 and parameters: {'alpha': 38.66492314396974}. Best is trial 1 with value: 11.012865424739477.
[I 2025-12-22 21:53:01,152] Trial 2 finished with value: 11.437154376830511 and parameters: {'alpha': 0.0458067051909611}. Best is trial 1 with value: 11.012865424739477.
[I 2025-12-22 21:53:01,194] Trial 3 finished with value: 12.407475699263296 and parameters: {'alpha': 6455.339504460167}. Best is trial 1 with value: 11.012865424739477.
[I 2025-12-22 21:53:01,234] Trial 4 finished with value: 12.311980823834975 and parameters: {'alpha': 1191.772516661805}. Best is trial 1 with value: 11.012865424739477.


Evaluating feature set (Ridge Regression): ['distPrim', 'gsv_fence_p1500', 'gsv_earth_p1500']


[I 2025-12-22 21:53:01,277] Trial 5 finished with value: 12.33097737098506 and parameters: {'alpha': 1430.6606496802312}. Best is trial 1 with value: 11.012865424739477.
[I 2025-12-22 21:53:01,327] Trial 6 finished with value: 12.3587899068982 and parameters: {'alpha': 2009.9872097757516}. Best is trial 1 with value: 11.012865424739477.
[I 2025-12-22 21:53:01,367] Trial 7 finished with value: 11.438111535574333 and parameters: {'alpha': 0.03795771739999753}. Best is trial 1 with value: 11.012865424739477.
[I 2025-12-22 21:53:01,407] Trial 8 finished with value: 11.79468778173065 and parameters: {'alpha': 175.04866823138167}. Best is trial 1 with value: 11.012865424739477.
[I 2025-12-22 21:53:01,466] Trial 9 finished with value: 11.347115071926751 and parameters: {'alpha': 75.91615573018932}. Best is trial 1 with value: 11.012865424739477.
[I 2025-12-22 21:53:01,514] Trial 10 finished with value: 10.982686332066269 and parameters: {'alpha': 6.592414258587711}. Best is trial 10 with valu

Best Parameters Found: {'alpha': 16.622038055775093}
Test RMSE: 6.7343

--- Generating Predictions for New Dataset ---

Head of Predictions:
      Predictions
OID_             
1268    57.362161
1271    54.467458
1272    52.716598
1273    38.758152
1274    39.836541

Prediction Statistics:
       Predictions
count  3724.000000
mean     57.322294
std       5.580788
min      37.477057
25%      54.005585
50%      57.100364
75%      60.206103
max      77.573949

Model Performance Metrics:
   RMSE_Train  R2_Train    RMSE_CV     R2_CV  RMSE_Test   R2_Test
0    9.023098  0.396285  10.847534  0.127466   6.734258  0.223726

Final Result Dictionary:
[{'features': ['distPrim', 'gsv_fence_p1500', 'gsv_earth_p1500'], 'best_params': {'alpha': 16.622038055775093}, 'model': Ridge(alpha=16.622038055775093, random_state=42), 'rmse_train': 9.023098491778837, 'r2_train': 0.3962852279967608, 'rmse_cv': 10.847534102413896, 'r2_cv': 0.1274658075022359, 'rmse_test': 6.734257716893187, 'r2_test': 0.22372593377

In [ ]:
import pandas as pd
import optuna
from sklearn.linear_model import Lasso
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import cross_val_predict

# Feature set to be used for prediction
features = ['distPrim', 'gsv_fence_p1500', 'gsv_earth_p1500']

# Function to run the optimization and evaluate the model
def evaluate_model(features):
    X_Train = X_st[features]
    X_test = X_test_st[features]

    def objective(trial):
        # Optimization parameter for Lasso: Alpha (Regularization strength)
        # Lasso alpha usually requires a search space similar to Ridge, sometimes slightly smaller/larger depending on sparsity
        alpha = trial.suggest_float('alpha', 0.0001, 100.0, log=True)

        # Instantiate Lasso model
        model = Lasso(alpha=alpha, random_state=42)

        predicted = cross_val_predict(model, X_Train, y_train, cv=10)
        # Minimize RMSE
        return mean_squared_error(y_train, predicted) ** 0.5

    study = optuna.create_study(direction='minimize', study_name="Lasso_RMSE")
    study.optimize(objective, n_trials=100) # 100 trials is sufficient for single-parameter tuning

    best_params = study.best_params

    # Rebuild final model with best params
    model = Lasso(
        alpha=best_params['alpha'],
        random_state=42
    )

    model.fit(X_Train, y_train)

    # --- Calculate True RMSE (Square root of MSE) ---

    # 1. Training Performance
    pred_train = model.predict(X_Train)
    rmse_train = mean_squared_error(y_train, pred_train) ** 0.5
    r2_train = r2_score(y_train, pred_train)

    # 2. Cross-Validation Performance
    predicted_cv = cross_val_predict(model, X_Train, y_train, cv=10)
    rmse_cv = mean_squared_error(y_train, predicted_cv) ** 0.5
    r2_cv = r2_score(y_train, predicted_cv)

    # 3. Test Performance
    predicted_test = model.predict(X_test)
    rmse_test = mean_squared_error(y_test, predicted_test) ** 0.5
    r2_test = r2_score(y_test, predicted_test)

    return {
        "features": features,
        "best_params": best_params,
        "model": model,
        "rmse_train": rmse_train,
        "r2_train": r2_train,
        "rmse_cv": rmse_cv,
        "r2_cv": r2_cv,
        "rmse_test": rmse_test,
        "r2_test": r2_test
    }

# Evaluate the model using the specified feature set
print(f"Evaluating feature set (Lasso Regression): {features}")

try:
    result = evaluate_model(features)
    results = [result]
    print("Best Parameters Found:", result["best_params"])
    print(f"Test RMSE: {result['rmse_test']:.4f}")

except ValueError as e:
    if "Input contains NaN" in str(e):
        print(f"Skipping evaluation due to NaN values in feature set: {features}")
        results = []
    else:
        raise e

# --- New Prediction & Saving Logic ---
if results:
    print("\n--- Generating Predictions for New Dataset ---")

    # Get the trained model
    model = results[0]['model']

    # Generate predictions for the new dataset
    x_pred_st_subset = x_pred_st[features]
    predictions = model.predict(x_pred_st_subset)

    # 1. Save Predictions
    predictions_df = pd.DataFrame(predictions, index=x_pred_st.index, columns=["Predictions"])
    predictions_df.to_csv("/content/drive/MyDrive/datasets_Rayeed_GSV_POI_LU/predictions_wet_model_pm25_lasso.csv")

    # 2. Save Prediction Stats
    stats = predictions_df.describe()
    stats.to_csv("/content/drive/MyDrive/datasets_Rayeed_GSV_POI_LU/predictions_wet_model_pm25_lasso_stats.csv")

    # 3. Save Model Performance Metrics (RMSE, R2)
    metrics_data = {
        "RMSE_Train": results[0]['rmse_train'],
        "R2_Train": results[0]['r2_train'],
        "RMSE_CV": results[0]['rmse_cv'],
        "R2_CV": results[0]['r2_cv'],
        "RMSE_Test": results[0]['rmse_test'],
        "R2_Test": results[0]['r2_test']
    }

    metrics_df = pd.DataFrame([metrics_data])
    metrics_df.to_csv("/content/drive/MyDrive/datasets_Rayeed_GSV_POI_LU/wet_model_lasso_performance_metrics.csv", index=False)

    print("\nHead of Predictions:")
    print(predictions_df.head())

    print("\nPrediction Statistics:")
    print(stats)

    print("\nModel Performance Metrics:")
    print(metrics_df)

    print("\nFinal Result Dictionary:")
    print(results)
else:
    print("No model was trained; skipping prediction generation.")

[I 2025-12-22 21:53:16,912] A new study created in memory with name: Lasso_RMSE
[I 2025-12-22 21:53:17,054] Trial 0 finished with value: 11.441211307399362 and parameters: {'alpha': 0.18672609171433896}. Best is trial 0 with value: 11.441211307399362.


Evaluating feature set (Lasso Regression): ['distPrim', 'gsv_fence_p1500', 'gsv_earth_p1500']


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
[I 2025-12-22 21:53:17,261] Trial 1 finished with value: 11.442577691548943 and parameters: {'alpha': 0.005536217102524286}. Best is trial 0 with value: 11.441211307399362.
[I 2025-12-22 21:53:17,366] Trial 2 finished with value: 11.442015687879683 and parameters: {'alpha': 0.023490772969894793}. Best is trial 0 with value: 11.441211307399362.
[I 2025-12-22 21:53:17,488] Trial 3 finished with value: 11.442489005482479 and parameters: {'alpha': 0.008181255822614506}. Best is trial 0 with value: 11.441211307399362.
[I 2025-12-22 21:53:17,729] Trial 4 finished with value: 11.442340785094766 and parameters: {'alpha': 0.012744647679616439}. Best is trial 0 with value: 11.4412

Best Parameters Found: {'alpha': 0.12054950514463346}
Test RMSE: 6.6966

--- Generating Predictions for New Dataset ---

Head of Predictions:
      Predictions
OID_             
1268    57.561105
1271    53.798562
1272    51.442928
1273    33.887092
1274    35.303382

Prediction Statistics:
       Predictions
count  3724.000000
mean     57.811483
std       7.843626
min      32.032849
25%      53.238193
50%      57.293089
75%      61.649921
max      88.260172

Model Performance Metrics:
   RMSE_Train  R2_Train    RMSE_CV     R2_CV  RMSE_Test   R2_Test
0    8.765444  0.430271  11.440602  0.029449   6.696596  0.232384

Final Result Dictionary:
[{'features': ['distPrim', 'gsv_fence_p1500', 'gsv_earth_p1500'], 'best_params': {'alpha': 0.12054950514463346}, 'model': Lasso(alpha=0.12054950514463346, random_state=42), 'rmse_train': 8.765443651989523, 'r2_train': 0.43027115093511736, 'rmse_cv': 11.440602380086597, 'r2_cv': 0.029449382092664345, 'rmse_test': 6.6965960717303465, 'r2_test': 0.2323

# ***Dry Model***

SVR

In [ ]:
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.preprocessing import KBinsDiscretizer

# Load the dataset
df = pd.read_excel('/content/drive/MyDrive/datasets_Rayeed_GSV_POI_LU/model_35_wet_dry.xlsx', index_col='site_name')

# Drop unnecessary columns
df.drop(['Site_type', 'OID'], inplace=True, axis=1)

# Separate the target variable and features
y = df['Mean_dry_pm']
X = df.drop(['Mean PNC (# / cm3)', 'Mean PM2.5 (µg/m3)','Mean_wet_pm','Mean_dry_pm','Mean_wet_pnc','Mean_dry_pnc'], axis=1)

In [ ]:
# Discretize the target variable into bins using 'quantile' strategy
binner = KBinsDiscretizer(n_bins=5, encode='ordinal', strategy='quantile')
y_binned = binner.fit_transform(y.values.reshape(-1, 1)).flatten()

# Perform stratified train-test split
split = StratifiedShuffleSplit(n_splits=1, test_size=.2, random_state=42)
for train_index, test_index in split.split(X, y_binned):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

# Check the distribution of bins in the train and test sets
print("Train bin distribution:\n", pd.value_counts(y_binned[train_index], sort=False))
print("Test bin distribution:\n", pd.value_counts(y_binned[test_index], sort=False))

# Output the resulting dataframes
print("X_train:\n", X_train.head())
print("y_train:\n", y_train.head())
print("X_test:\n", X_test.head())
print("y_test:\n", y_test.head())

Train bin distribution:
 0.0    5
1.0    6
3.0    5
4.0    6
2.0    6
Name: count, dtype: int64
Test bin distribution:
 0.0    2
1.0    1
3.0    2
2.0    1
4.0    1
Name: count, dtype: int64
X_train:
                  gsv_wall_p100  gsv_building_p100  gsv_house_p100  \
site_name                                                           
Romna_park             23.3075             0.0000          0.0000   
Eskaton                 5.7200            26.1825          0.0000   
US_Embassy              3.2375             2.4300          0.0000   
Aftabnagar              3.6900            32.4275          0.2575   
Square_hospital        59.2475             0.0000          0.0000   

                 gsv_awning_p100  gsv_sky_p100  gsv_earth_p100  gsv_tree_p100  \
site_name                                                                       
Romna_park                   0.0        0.0150          0.0000         0.0000   
Eskaton                      0.0       19.5250          0.0000        23

/tmp/ipython-input-3543545133.py:12: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  print("Train bin distribution:\n", pd.value_counts(y_binned[train_index], sort=False))
/tmp/ipython-input-3543545133.py:13: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  print("Test bin distribution:\n", pd.value_counts(y_binned[test_index], sort=False))


In [ ]:
# Discretize the target variable into bins using 'quantile' strategy
binner = KBinsDiscretizer(n_bins=5, encode='ordinal', strategy='quantile')
y_binned = binner.fit_transform(y.values.reshape(-1, 1)).flatten()

# Perform stratified train-test split
split = StratifiedShuffleSplit(n_splits=1, test_size=.2, random_state=42)
for train_index, test_index in split.split(X, y_binned):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

# Check the distribution of bins in the train and test sets
print("Train bin distribution:\n", pd.value_counts(y_binned[train_index], sort=False))
print("Test bin distribution:\n", pd.value_counts(y_binned[test_index], sort=False))

# Output the resulting dataframes
print("X_train:\n", X_train.head())
print("y_train:\n", y_train.head())
print("X_test:\n", X_test.head())
print("y_test:\n", y_test.head())

Train bin distribution:
 0.0    5
1.0    6
3.0    5
4.0    6
2.0    6
Name: count, dtype: int64
Test bin distribution:
 0.0    2
1.0    1
3.0    2
2.0    1
4.0    1
Name: count, dtype: int64
X_train:
                  gsv_wall_p100  gsv_building_p100  gsv_house_p100  \
site_name                                                           
Romna_park             23.3075             0.0000          0.0000   
Eskaton                 5.7200            26.1825          0.0000   
US_Embassy              3.2375             2.4300          0.0000   
Aftabnagar              3.6900            32.4275          0.2575   
Square_hospital        59.2475             0.0000          0.0000   

                 gsv_awning_p100  gsv_sky_p100  gsv_earth_p100  gsv_tree_p100  \
site_name                                                                       
Romna_park                   0.0        0.0150          0.0000         0.0000   
Eskaton                      0.0       19.5250          0.0000        23

/tmp/ipython-input-3543545133.py:12: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  print("Train bin distribution:\n", pd.value_counts(y_binned[train_index], sort=False))
/tmp/ipython-input-3543545133.py:13: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  print("Test bin distribution:\n", pd.value_counts(y_binned[test_index], sort=False))


In [ ]:
# Apply Kolmogorov-Smirnov test for each feature
ks_results = {}
failed_features = []
alpha = 0.05
for feature in X.columns:
    ks_statistic, p_value = ks_2samp(X_train[feature], X_test[feature])
    ks_results[feature] = {'ks_statistic': ks_statistic, 'p_value': p_value}
    if p_value < alpha:
        failed_features.append(feature)

# Print K-S test results for each feature
for feature, result in ks_results.items():
    print(f"Feature: {feature}")
    print(f"K-S statistic: {result['ks_statistic']}")
    print(f"P-value: {result['p_value']}")
    if result['p_value'] < alpha:
        print("The null hypothesis is rejected. The distributions of the train and test sets are different for this feature.\n")
    else:
        print("The null hypothesis cannot be rejected. The distributions of the train and test sets are the same for this feature.\n")

# Print features that failed the K-S test
if failed_features:
    print("Features that failed the K-S test (null hypothesis rejected):")
    for feature in failed_features:
        print(f"- {feature}")
else:
    print("All features passed the K-S test (null hypothesis not rejected).")

Feature: gsv_wall_p100
K-S statistic: 0.32142857142857145
P-value: 0.5556032252116138
The null hypothesis cannot be rejected. The distributions of the train and test sets are the same for this feature.

Feature: gsv_building_p100
K-S statistic: 0.42857142857142855
P-value: 0.21839938017880833
The null hypothesis cannot be rejected. The distributions of the train and test sets are the same for this feature.

Feature: gsv_house_p100
K-S statistic: 0.2857142857142857
P-value: 0.7050303962215889
The null hypothesis cannot be rejected. The distributions of the train and test sets are the same for this feature.

Feature: gsv_awning_p100
K-S statistic: 0.32142857142857145
P-value: 0.5556032252116138
The null hypothesis cannot be rejected. The distributions of the train and test sets are the same for this feature.

Feature: gsv_sky_p100
K-S statistic: 0.32142857142857145
P-value: 0.5556032252116138
The null hypothesis cannot be rejected. The distributions of the train and test sets are the sam

In [ ]:
X_train_dropped = X_train.drop(columns=failed_features)
X_test_dropped = X_test.drop(columns=failed_features)

trans = StandardScaler()
X_st = trans.fit_transform(X_train_dropped)
X_st = pd.DataFrame(X_st, columns=X_train_dropped.columns, index=X_train_dropped.index)

X_test_st = trans.transform(X_test_dropped)
X_test_st = pd.DataFrame(X_test_st, columns=X_test_dropped.columns, index=X_test_dropped.index)


In [ ]:
import pandas as pd
import optuna
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import cross_val_predict

# Define the list of feature sets to iterate through
feature_sets = [
    ['gsv_path_p1500', 'gsv_sidewalk_p500', 'gsv_fence_p1500', 'gsv_wall_p250'],
    ['gsv_path_p1500', 'gsv_sidewalk_p500', 'gsv_fence_p1500', 'land_Ar_MC_p_750'],
    ['gsv_path_p1500', 'gsv_sidewalk_p500', 'gsv_fence_p1500', 'gsv_tree_p1000'],
    ['gsv_path_p1500', 'gsv_sidewalk_p500', 'gsv_house_p500', 'gsv_fence_p1500']
]

def evaluate_model(features, set_index):
    # Create local subsets for this specific feature set
    X_Train = X_st[features]
    X_test = X_test_st[features]

    def objective(trial):
        # 1. Fixed Kernel (Linear as per previous request)
        kernel = 'linear'

        # 2. Optimize Hyperparameters
        C = trial.suggest_float("C", 0.01, 1000000.0, log=True)
        epsilon = trial.suggest_float("epsilon", 0.001, 10.0)

        # 3. Train and Evaluate
        svr = SVR(kernel=kernel, C=C, epsilon=epsilon)

        predicted = cross_val_predict(svr, X_Train, y_train, cv=10)
        return mean_squared_error(y_train, predicted)

    # Unique study name for each set to avoid conflicts
    study_name = f"SVR_Linear_RMSE_Set_{set_index}"
    study = optuna.create_study(direction='minimize', study_name=study_name)
    study.optimize(objective, n_trials=100)

    # --- Rebuild Final Model ---
    best_params = study.best_params

    model = SVR(
        kernel='linear',
        C=best_params['C'],
        epsilon=best_params['epsilon']
    )

    model.fit(X_Train, y_train)

    # --- Calculate True RMSE ---
    pred_train = model.predict(X_Train)
    rmse_train = mean_squared_error(y_train, pred_train) ** 0.5
    r2_train = r2_score(y_train, pred_train)

    predicted_cv = cross_val_predict(model, X_Train, y_train, cv=10)
    rmse_cv = mean_squared_error(y_train, predicted_cv) ** 0.5
    r2_cv = r2_score(y_train, predicted_cv)

    predicted_test = model.predict(X_test)
    rmse_test = mean_squared_error(y_test, predicted_test) ** 0.5
    r2_test = r2_score(y_test, predicted_test)

    return {
        "set_index": set_index,
        "features": features,
        "best_params": best_params,
        "model": model,
        "rmse_train": rmse_train,
        "r2_train": r2_train,
        "rmse_cv": rmse_cv,
        "r2_cv": r2_cv,
        "rmse_test": rmse_test,
        "r2_test": r2_test
    }

# --- Main Execution Loop ---
all_results = []

for i, features in enumerate(feature_sets, 1):
    print(f"\n{'='*60}")
    print(f"Processing Feature Set {i}: {features}")
    print(f"{'='*60}")

    try:
        result = evaluate_model(features, i)
        all_results.append(result)

        print(f"Set {i} Best Params: {result['best_params']}")
        print(f"Set {i} Test RMSE: {result['rmse_test']:.4f}")

        # --- Generate and Save Predictions for this Set ---
        model = result['model']
        x_pred_st_subset = x_pred_st[features]
        predictions = model.predict(x_pred_st_subset)

        # File names include 'dry_model' and the set number
        base_path = "/content/drive/MyDrive/datasets_Rayeed_GSV_POI_LU/"

        # 1. Save Predictions
        pred_filename = f"predictions_dry_model_set_{i}_svr.csv"
        predictions_df = pd.DataFrame(predictions, index=x_pred_st.index, columns=["Predictions"])
        predictions_df.to_csv(base_path + pred_filename)

        # 2. Save Prediction Stats
        stats_filename = f"predictions_dry_model_set_{i}_svr_stats.csv"
        stats = predictions_df.describe()
        stats.to_csv(base_path + stats_filename)

        # 3. Save Performance Metrics
        metrics_filename = f"dry_model_set_{i}_svr_performance_metrics.csv"
        metrics_data = {
            "Feature_Set_Index": i,
            "Features": str(features),
            "RMSE_Train": result['rmse_train'],
            "R2_Train": result['r2_train'],
            "RMSE_CV": result['rmse_cv'],
            "R2_CV": result['r2_cv'],
            "RMSE_Test": result['rmse_test'],
            "R2_Test": result['r2_test']
        }
        metrics_df = pd.DataFrame([metrics_data])
        metrics_df.to_csv(base_path + metrics_filename, index=False)

        print(f"--> Saved results for Set {i} to {base_path}")

    except ValueError as e:
        if "Input contains NaN" in str(e):
            print(f"Skipping Set {i} due to NaN values in features.")
        else:
            raise e

print("\nAll feature sets processed.")

[I 2025-12-21 20:55:20,761] A new study created in memory with name: SVR_Linear_RMSE_Set_1
[I 2025-12-21 20:55:20,856] Trial 0 finished with value: 1361.5057804835858 and parameters: {'C': 0.025884854344407536, 'epsilon': 3.632531312706698}. Best is trial 0 with value: 1361.5057804835858.
[I 2025-12-21 20:55:20,931] Trial 1 finished with value: 1371.1920003786488 and parameters: {'C': 0.0900888141649544, 'epsilon': 9.056976242727025}. Best is trial 0 with value: 1361.5057804835858.



Processing Feature Set 1: ['gsv_path_p1500', 'gsv_sidewalk_p500', 'gsv_fence_p1500', 'gsv_wall_p250']


[I 2025-12-21 20:55:21,993] Trial 2 finished with value: 637.1983134454613 and parameters: {'C': 15245.638688793906, 'epsilon': 2.093850277611438}. Best is trial 2 with value: 637.1983134454613.
[I 2025-12-21 20:55:22,067] Trial 3 finished with value: 919.3917093194902 and parameters: {'C': 0.6178022310308916, 'epsilon': 4.328949579476647}. Best is trial 2 with value: 637.1983134454613.
[I 2025-12-21 20:55:22,163] Trial 4 finished with value: 567.0287630436011 and parameters: {'C': 776.6228496021737, 'epsilon': 9.205427472423715}. Best is trial 4 with value: 567.0287630436011.
[I 2025-12-21 20:55:28,339] Trial 5 finished with value: 567.3332976467449 and parameters: {'C': 119516.28331096206, 'epsilon': 9.217350760522576}. Best is trial 4 with value: 567.0287630436011.
[I 2025-12-21 20:55:28,525] Trial 6 finished with value: 650.2901038292937 and parameters: {'C': 1461.955394364992, 'epsilon': 0.4159579781070173}. Best is trial 4 with value: 567.0287630436011.
[I 2025-12-21 20:55:32,861

Set 1 Best Params: {'C': 9.275566358380182, 'epsilon': 8.782046690595307}
Set 1 Test RMSE: 21.2388
--> Saved results for Set 1 to /content/drive/MyDrive/datasets_Rayeed_GSV_POI_LU/

Processing Feature Set 2: ['gsv_path_p1500', 'gsv_sidewalk_p500', 'gsv_fence_p1500', 'land_Ar_MC_p_750']


[I 2025-12-21 20:56:37,461] Trial 2 finished with value: 1883.560678470217 and parameters: {'C': 41423.83640424117, 'epsilon': 8.903251124624186}. Best is trial 1 with value: 1245.6712287644696.
[I 2025-12-21 20:56:37,518] Trial 3 finished with value: 1381.2197677425218 and parameters: {'C': 0.016219272259026408, 'epsilon': 2.644304557849432}. Best is trial 1 with value: 1245.6712287644696.
[I 2025-12-21 20:56:37,574] Trial 4 finished with value: 1316.5021290987263 and parameters: {'C': 0.07843386591409589, 'epsilon': 3.891026901562251}. Best is trial 1 with value: 1245.6712287644696.
[I 2025-12-21 20:56:38,081] Trial 5 finished with value: 1709.9190655760697 and parameters: {'C': 9202.01044005848, 'epsilon': 3.4456656542874065}. Best is trial 1 with value: 1245.6712287644696.
[I 2025-12-21 20:56:38,149] Trial 6 finished with value: 606.6317575557954 and parameters: {'C': 256.8457232061547, 'epsilon': 1.2378411433370153}. Best is trial 6 with value: 606.6317575557954.
[I 2025-12-21 20:

Set 2 Best Params: {'C': 11.98706696643872, 'epsilon': 3.377269936890305}
Set 2 Test RMSE: 22.6180
--> Saved results for Set 2 to /content/drive/MyDrive/datasets_Rayeed_GSV_POI_LU/

Processing Feature Set 3: ['gsv_path_p1500', 'gsv_sidewalk_p500', 'gsv_fence_p1500', 'gsv_tree_p1000']


[I 2025-12-21 20:57:13,057] Trial 2 finished with value: 964.2165789236475 and parameters: {'C': 0.3683940814339573, 'epsilon': 2.9366558311111004}. Best is trial 1 with value: 635.496585275945.
[I 2025-12-21 20:57:13,109] Trial 3 finished with value: 644.6388788665391 and parameters: {'C': 33.869985254558856, 'epsilon': 9.265394709952764}. Best is trial 1 with value: 635.496585275945.
[I 2025-12-21 20:57:13,162] Trial 4 finished with value: 606.3992356748378 and parameters: {'C': 3.385752441228672, 'epsilon': 5.918754134873147}. Best is trial 4 with value: 606.3992356748378.
[I 2025-12-21 20:57:13,205] Trial 5 finished with value: 731.3823939639848 and parameters: {'C': 68.56515322011484, 'epsilon': 3.917055737085239}. Best is trial 4 with value: 606.3992356748378.
[I 2025-12-21 20:57:13,247] Trial 6 finished with value: 1453.2496898446987 and parameters: {'C': 0.012030951929231405, 'epsilon': 8.738049155637658}. Best is trial 4 with value: 606.3992356748378.
[I 2025-12-21 20:57:13,45

Set 3 Best Params: {'C': 5.959567492351048, 'epsilon': 7.826099160223322}
Set 3 Test RMSE: 22.0185
--> Saved results for Set 3 to /content/drive/MyDrive/datasets_Rayeed_GSV_POI_LU/

Processing Feature Set 4: ['gsv_path_p1500', 'gsv_sidewalk_p500', 'gsv_house_p500', 'gsv_fence_p1500']


[I 2025-12-21 20:57:41,831] Trial 2 finished with value: 621.5484781086808 and parameters: {'C': 44017.4302866578, 'epsilon': 8.198798675796894}. Best is trial 1 with value: 589.165352888299.
[I 2025-12-21 20:57:43,638] Trial 3 finished with value: 630.1301845932064 and parameters: {'C': 61708.01065826239, 'epsilon': 1.6464320491220794}. Best is trial 1 with value: 589.165352888299.
[I 2025-12-21 20:57:43,682] Trial 4 finished with value: 578.1041879431261 and parameters: {'C': 3.5726338930293466, 'epsilon': 0.4831275400021396}. Best is trial 4 with value: 578.1041879431261.
[I 2025-12-21 20:57:43,726] Trial 5 finished with value: 651.5663756029884 and parameters: {'C': 186.55166981048055, 'epsilon': 5.954758763446498}. Best is trial 4 with value: 578.1041879431261.
[I 2025-12-21 20:57:43,774] Trial 6 finished with value: 625.6786662089606 and parameters: {'C': 249.44955554960023, 'epsilon': 1.889273125608963}. Best is trial 4 with value: 578.1041879431261.
[I 2025-12-21 20:57:43,816] 

Set 4 Best Params: {'C': 4.851636050303548, 'epsilon': 6.36441156776855}
Set 4 Test RMSE: 22.6860
--> Saved results for Set 4 to /content/drive/MyDrive/datasets_Rayeed_GSV_POI_LU/

All feature sets processed.


Linear regression

In [ ]:
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import cross_val_predict

# Define the list of feature sets to iterate through
feature_sets = [
    ['gsv_path_p1500', 'gsv_sidewalk_p500', 'gsv_fence_p1500', 'gsv_wall_p250'],
    ['gsv_path_p1500', 'gsv_sidewalk_p500', 'gsv_fence_p1500', 'land_Ar_MC_p_750'],
    ['gsv_path_p1500', 'gsv_sidewalk_p500', 'gsv_fence_p1500', 'gsv_tree_p1000'],
    ['gsv_path_p1500', 'gsv_sidewalk_p500', 'gsv_house_p500', 'gsv_fence_p1500']
]

def evaluate_linear_model(features, set_index):
    # Create local subsets for this specific feature set
    X_Train = X_st[features]
    X_test = X_test_st[features]

    # Initialize and Fit Linear Regression (No hyperparameters to tune)
    model = LinearRegression()
    model.fit(X_Train, y_train)

    # --- Calculate Metrics (RMSE = sqrt(MSE)) ---

    # 1. Training
    pred_train = model.predict(X_Train)
    rmse_train = mean_squared_error(y_train, pred_train) ** 0.5
    r2_train = r2_score(y_train, pred_train)

    # 2. Cross-Validation
    predicted_cv = cross_val_predict(model, X_Train, y_train, cv=10)
    rmse_cv = mean_squared_error(y_train, predicted_cv) ** 0.5
    r2_cv = r2_score(y_train, predicted_cv)

    # 3. Testing
    predicted_test = model.predict(X_test)
    rmse_test = mean_squared_error(y_test, predicted_test) ** 0.5
    r2_test = r2_score(y_test, predicted_test)

    return {
        "Set_Index": set_index,
        "Features": str(features),
        "Model": model,
        "RMSE_Train": rmse_train,
        "R2_Train": r2_train,
        "RMSE_CV": rmse_cv,
        "R2_CV": r2_cv,
        "RMSE_Test": rmse_test,
        "R2_Test": r2_test
    }

# --- Main Execution Loop ---
combined_metrics = []
base_path = "/content/drive/MyDrive/datasets_Rayeed_GSV_POI_LU/"

for i, features in enumerate(feature_sets, 1):
    print(f"\n{'='*60}")
    print(f"Processing Feature Set {i}: {features}")
    print(f"{'='*60}")

    try:
        result = evaluate_linear_model(features, i)

        # Add metrics to the combined list (excluding the model object itself)
        metric_record = {k: v for k, v in result.items() if k != 'Model'}
        combined_metrics.append(metric_record)

        print(f"Set {i} Test RMSE: {result['RMSE_Test']:.4f}")

        # --- Generate and Save Predictions for this Set ---
        model = result['Model']
        x_pred_st_subset = x_pred_st[features]
        predictions = model.predict(x_pred_st_subset)

        # 1. Save Predictions
        pred_filename = f"predictions_dry_model_set_{i}_linear.csv"
        predictions_df = pd.DataFrame(predictions, index=x_pred_st.index, columns=["Predictions"])
        predictions_df.to_csv(base_path + pred_filename)

        # 2. Save Prediction Stats
        stats_filename = f"predictions_dry_model_set_{i}_linear_stats.csv"
        stats = predictions_df.describe()
        stats.to_csv(base_path + stats_filename)

        print(f"--> Saved predictions for Set {i}")

    except ValueError as e:
        if "Input contains NaN" in str(e):
            print(f"Skipping Set {i} due to NaN values in features.")
        else:
            raise e

# --- Save Combined Performance Metrics ---
if combined_metrics:
    print(f"\n{'='*60}")
    print("Saving Combined Performance Metrics...")

    metrics_df = pd.DataFrame(combined_metrics)

    # Save to a single CSV file
    combined_filename = "dry_model_linear_combined_performance_metrics.csv"
    metrics_df.to_csv(base_path + combined_filename, index=False)

    print(f"Successfully saved combined metrics to: {base_path + combined_filename}")
    print(metrics_df)
else:
    print("No metrics were generated.")


Processing Feature Set 1: ['gsv_path_p1500', 'gsv_sidewalk_p500', 'gsv_fence_p1500', 'gsv_wall_p250']
Set 1 Test RMSE: 21.5061
--> Saved predictions for Set 1

Processing Feature Set 2: ['gsv_path_p1500', 'gsv_sidewalk_p500', 'gsv_fence_p1500', 'land_Ar_MC_p_750']
Set 2 Test RMSE: 21.6951
--> Saved predictions for Set 2

Processing Feature Set 3: ['gsv_path_p1500', 'gsv_sidewalk_p500', 'gsv_fence_p1500', 'gsv_tree_p1000']
Set 3 Test RMSE: 21.9038
--> Saved predictions for Set 3

Processing Feature Set 4: ['gsv_path_p1500', 'gsv_sidewalk_p500', 'gsv_house_p500', 'gsv_fence_p1500']
Set 4 Test RMSE: 22.8564
--> Saved predictions for Set 4

Saving Combined Performance Metrics...
Successfully saved combined metrics to: /content/drive/MyDrive/datasets_Rayeed_GSV_POI_LU/dry_model_linear_combined_performance_metrics.csv
   Set_Index                                           Features  RMSE_Train  \
0          1  ['gsv_path_p1500', 'gsv_sidewalk_p500', 'gsv_f...   20.198238   
1          2  ['g

xgboost

In [ ]:
import pandas as pd
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import cross_val_predict
from xgboost import XGBRegressor
import optuna

# Define the list of feature sets to iterate through
feature_sets = [
    ['gsv_path_p1500', 'gsv_sidewalk_p500', 'gsv_fence_p1500', 'gsv_wall_p250'],
    ['gsv_path_p1500', 'gsv_sidewalk_p500', 'gsv_fence_p1500', 'land_Ar_MC_p_750'],
    ['gsv_path_p1500', 'gsv_sidewalk_p500', 'gsv_fence_p1500', 'gsv_tree_p1000'],
    ['gsv_path_p1500', 'gsv_sidewalk_p500', 'gsv_house_p500', 'gsv_fence_p1500']
]

def evaluate_xgb_model(features, set_index):
    X_Train = X_st[features]
    X_test = X_test_st[features]

    def objective(trial):
        # Optimization parameters
        learning_rate = trial.suggest_float('learning_rate', 0.005, 1.5)
        n_estimators = trial.suggest_int('n_estimators', 50, 1000)
        reg_alpha = trial.suggest_float('reg_alpha', 0.001, 100)
        reg_lambda = trial.suggest_float('reg_lambda', 0.001, 3)

        XGB = XGBRegressor(
            booster='gblinear',
            n_estimators=n_estimators,
            learning_rate=learning_rate,
            reg_alpha=reg_alpha,
            reg_lambda=reg_lambda,
            objective="reg:squarederror",
            seed=42,
            verbosity=0
        )

        predicted = cross_val_predict(XGB, X_Train, y_train, cv=10)
        return mean_squared_error(y_train, predicted) ** 0.5

    # Unique study name to avoid conflicts
    study_name = f"XGB_RMSE_Set_{set_index}"
    study = optuna.create_study(direction='minimize', study_name=study_name)
    study.optimize(objective, n_trials=200) # 200 trials per set

    best_params = study.best_params

    # Rebuild final model with best params
    model = XGBRegressor(
        booster='gblinear',
        n_estimators=best_params['n_estimators'],
        learning_rate=best_params['learning_rate'],
        reg_alpha=best_params['reg_alpha'],
        reg_lambda=best_params['reg_lambda'],
        objective="reg:squarederror",
        seed=42
    )

    model.fit(X_Train, y_train)

    # --- Calculate True RMSE ---
    pred_train = model.predict(X_Train)
    rmse_train = mean_squared_error(y_train, pred_train) ** 0.5
    r2_train = r2_score(y_train, pred_train)

    predicted_cv = cross_val_predict(model, X_Train, y_train, cv=10)
    rmse_cv = mean_squared_error(y_train, predicted_cv) ** 0.5
    r2_cv = r2_score(y_train, predicted_cv)

    predicted_test = model.predict(X_test)
    rmse_test = mean_squared_error(y_test, predicted_test) ** 0.5
    r2_test = r2_score(y_test, predicted_test)

    return {
        "Set_Index": set_index,
        "Features": str(features),
        "Best_Params": str(best_params),
        "Model": model,
        "RMSE_Train": rmse_train,
        "R2_Train": r2_train,
        "RMSE_CV": rmse_cv,
        "R2_CV": r2_cv,
        "RMSE_Test": rmse_test,
        "R2_Test": r2_test
    }

# --- Main Execution Loop ---
combined_metrics = []
base_path = "/content/drive/MyDrive/datasets_Rayeed_GSV_POI_LU/"

for i, features in enumerate(feature_sets, 1):
    print(f"\n{'='*60}")
    print(f"Processing Feature Set {i}: {features}")
    print(f"{'='*60}")

    try:
        result = evaluate_xgb_model(features, i)

        # Add metrics to the combined list (excluding the model object)
        metric_record = {k: v for k, v in result.items() if k != 'Model'}
        combined_metrics.append(metric_record)

        print(f"Set {i} Best Params: {result['Best_Params']}")
        print(f"Set {i} Test RMSE: {result['RMSE_Test']:.4f}")

        # --- Generate and Save Predictions ---
        model = result['Model']
        x_pred_st_subset = x_pred_st[features]
        predictions = model.predict(x_pred_st_subset)

        # 1. Save Predictions
        pred_filename = f"predictions_dry_model_set_{i}_xgb.csv"
        predictions_df = pd.DataFrame(predictions, index=x_pred_st.index, columns=["Predictions"])
        predictions_df.to_csv(base_path + pred_filename)

        # 2. Save Prediction Stats
        stats_filename = f"predictions_dry_model_set_{i}_xgb_stats.csv"
        stats = predictions_df.describe()
        stats.to_csv(base_path + stats_filename)

        print(f"--> Saved predictions for Set {i}")

    except ValueError as e:
        if "Input contains NaN" in str(e):
            print(f"Skipping Set {i} due to NaN values in features.")
        else:
            raise e

# --- Save Combined Performance Metrics ---
if combined_metrics:
    print(f"\n{'='*60}")
    print("Saving Combined Performance Metrics...")

    metrics_df = pd.DataFrame(combined_metrics)

    # Save to a single CSV file
    combined_filename = "dry_model_xgb_combined_performance_metrics.csv"
    metrics_df.to_csv(base_path + combined_filename, index=False)

    print(f"Successfully saved combined metrics to: {base_path + combined_filename}")
    print(metrics_df)
else:
    print("No metrics were generated.")

[I 2025-12-21 21:02:31,099] A new study created in memory with name: XGB_RMSE_Set_1



Processing Feature Set 1: ['gsv_path_p1500', 'gsv_sidewalk_p500', 'gsv_fence_p1500', 'gsv_wall_p250']


[I 2025-12-21 21:02:34,181] Trial 0 finished with value: 38.07827206069116 and parameters: {'learning_rate': 1.1372673212580375, 'n_estimators': 995, 'reg_alpha': 20.530325490114603, 'reg_lambda': 2.1112285487211553}. Best is trial 0 with value: 38.07827206069116.
[I 2025-12-21 21:02:34,702] Trial 1 finished with value: 38.199610411996616 and parameters: {'learning_rate': 0.17716133861638414, 'n_estimators': 334, 'reg_alpha': 97.90071651062617, 'reg_lambda': 0.5594490776873714}. Best is trial 0 with value: 38.07827206069116.
[I 2025-12-21 21:02:36,423] Trial 2 finished with value: 37.68592465471223 and parameters: {'learning_rate': 1.4715928592266905, 'n_estimators': 639, 'reg_alpha': 17.16389479553683, 'reg_lambda': 2.0029893955705584}. Best is trial 2 with value: 37.68592465471223.
[I 2025-12-21 21:02:37,670] Trial 3 finished with value: 38.199610352857135 and parameters: {'learning_rate': 1.3506103667378335, 'n_estimators': 992, 'reg_alpha': 50.23352883790308, 'reg_lambda': 2.325682

Set 1 Best Params: {'learning_rate': 0.0629675779799128, 'n_estimators': 558, 'reg_alpha': 0.006304293787904758, 'reg_lambda': 0.007012191510904488}
Set 1 Test RMSE: 21.5009
--> Saved predictions for Set 1

Processing Feature Set 2: ['gsv_path_p1500', 'gsv_sidewalk_p500', 'gsv_fence_p1500', 'land_Ar_MC_p_750']


[I 2025-12-21 21:05:09,876] Trial 0 finished with value: 38.19961094384975 and parameters: {'learning_rate': 1.4406447855895008, 'n_estimators': 1000, 'reg_alpha': 34.363021038873356, 'reg_lambda': 2.7847039178820334}. Best is trial 0 with value: 38.19961094384975.
[I 2025-12-21 21:05:10,398] Trial 1 finished with value: 27.23271723103648 and parameters: {'learning_rate': 0.6734752020670364, 'n_estimators': 918, 'reg_alpha': 4.619828334374989, 'reg_lambda': 0.20599522840353782}. Best is trial 1 with value: 27.23271723103648.
[I 2025-12-21 21:05:10,921] Trial 2 finished with value: 38.20525335794832 and parameters: {'learning_rate': 1.290897163198387, 'n_estimators': 898, 'reg_alpha': 24.18484239404955, 'reg_lambda': 2.581044073034868}. Best is trial 1 with value: 27.23271723103648.
[I 2025-12-21 21:05:11,042] Trial 3 finished with value: 38.199613906727095 and parameters: {'learning_rate': 1.0347383408060395, 'n_estimators': 91, 'reg_alpha': 87.83275047575806, 'reg_lambda': 0.692024489

Set 2 Best Params: {'learning_rate': 0.32637797653955186, 'n_estimators': 691, 'reg_alpha': 0.10785812835529958, 'reg_lambda': 0.14937001489509072}
Set 2 Test RMSE: 21.5790
--> Saved predictions for Set 2

Processing Feature Set 3: ['gsv_path_p1500', 'gsv_sidewalk_p500', 'gsv_fence_p1500', 'gsv_tree_p1000']


[I 2025-12-21 21:06:37,342] Trial 0 finished with value: 38.1996147800568 and parameters: {'learning_rate': 1.3044299931779073, 'n_estimators': 714, 'reg_alpha': 78.55940877084878, 'reg_lambda': 0.6685741837288145}. Best is trial 0 with value: 38.1996147800568.
[I 2025-12-21 21:06:37,666] Trial 1 finished with value: 38.199614675866336 and parameters: {'learning_rate': 1.2661662682510912, 'n_estimators': 504, 'reg_alpha': 78.12196450392439, 'reg_lambda': 2.713866405259015}. Best is trial 1 with value: 38.199614675866336.
[I 2025-12-21 21:06:38,061] Trial 2 finished with value: 38.1996129927854 and parameters: {'learning_rate': 1.3739005325667748, 'n_estimators': 644, 'reg_alpha': 78.53565001480374, 'reg_lambda': 0.2253222318454902}. Best is trial 2 with value: 38.1996129927854.
[I 2025-12-21 21:06:38,506] Trial 3 finished with value: 38.19961586220049 and parameters: {'learning_rate': 1.117276319357827, 'n_estimators': 697, 'reg_alpha': 36.134715216496744, 'reg_lambda': 1.0654294653597

Skipping Set 3 due to NaN values in features.

Processing Feature Set 4: ['gsv_path_p1500', 'gsv_sidewalk_p500', 'gsv_house_p500', 'gsv_fence_p1500']


[I 2025-12-21 21:07:08,003] Trial 0 finished with value: 34.00699664590354 and parameters: {'learning_rate': 0.34488595229428465, 'n_estimators': 717, 'reg_alpha': 11.05135755158251, 'reg_lambda': 0.7961428656386123}. Best is trial 0 with value: 34.00699664590354.
[I 2025-12-21 21:07:08,128] Trial 1 finished with value: 38.19961394950341 and parameters: {'learning_rate': 0.9216898097407866, 'n_estimators': 95, 'reg_alpha': 46.65852646214306, 'reg_lambda': 1.5017309652358792}. Best is trial 0 with value: 34.00699664590354.
[I 2025-12-21 21:07:08,380] Trial 2 finished with value: 38.19961346042663 and parameters: {'learning_rate': 0.19489873090584595, 'n_estimators': 371, 'reg_alpha': 60.52207258852392, 'reg_lambda': 0.8378350575879805}. Best is trial 0 with value: 34.00699664590354.
[I 2025-12-21 21:07:08,938] Trial 3 finished with value: 38.199612847210744 and parameters: {'learning_rate': 0.3565831736855578, 'n_estimators': 989, 'reg_alpha': 29.45360395599492, 'reg_lambda': 0.76566255

Set 4 Best Params: {'learning_rate': 0.11775735844230317, 'n_estimators': 757, 'reg_alpha': 0.004797159848792695, 'reg_lambda': 0.011395995006655767}
Set 4 Test RMSE: 22.9326
--> Saved predictions for Set 4

Saving Combined Performance Metrics...
Successfully saved combined metrics to: /content/drive/MyDrive/datasets_Rayeed_GSV_POI_LU/dry_model_xgb_combined_performance_metrics.csv
   Set_Index                                           Features  \
0          1  ['gsv_path_p1500', 'gsv_sidewalk_p500', 'gsv_f...   
1          2  ['gsv_path_p1500', 'gsv_sidewalk_p500', 'gsv_f...   
2          4  ['gsv_path_p1500', 'gsv_sidewalk_p500', 'gsv_h...   

                                         Best_Params  RMSE_Train  R2_Train  \
0  {'learning_rate': 0.0629675779799128, 'n_estim...   20.199155  0.684735   
1  {'learning_rate': 0.32637797653955186, 'n_esti...   20.443671  0.677056   
2  {'learning_rate': 0.11775735844230317, 'n_esti...   20.169195  0.685669   

     RMSE_CV     R2_CV  RMSE_Test 

ridge

In [ ]:
import pandas as pd
import optuna
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import cross_val_predict

# Define the list of feature sets to iterate through
feature_sets = [
    ['gsv_path_p1500', 'gsv_sidewalk_p500', 'gsv_fence_p1500', 'gsv_wall_p250'],
    ['gsv_path_p1500', 'gsv_sidewalk_p500', 'gsv_fence_p1500', 'land_Ar_MC_p_750'],
    ['gsv_path_p1500', 'gsv_sidewalk_p500', 'gsv_fence_p1500', 'gsv_tree_p1000'],
    ['gsv_path_p1500', 'gsv_sidewalk_p500', 'gsv_house_p500', 'gsv_fence_p1500']
]

def evaluate_ridge_model(features, set_index):
    X_Train = X_st[features]
    X_test = X_test_st[features]

    def objective(trial):
        # Optimization parameter for Ridge: Alpha
        alpha = trial.suggest_float('alpha', 0.01, 10000.0, log=True)

        model = Ridge(alpha=alpha, random_state=42)

        predicted = cross_val_predict(model, X_Train, y_train, cv=10)
        return mean_squared_error(y_train, predicted) ** 0.5

    # Unique study name
    study_name = f"Ridge_RMSE_Set_{set_index}"
    study = optuna.create_study(direction='minimize', study_name=study_name)
    study.optimize(objective, n_trials=100)

    best_params = study.best_params

    # Rebuild final model
    model = Ridge(
        alpha=best_params['alpha'],
        random_state=42
    )

    model.fit(X_Train, y_train)

    # --- Calculate True RMSE ---
    pred_train = model.predict(X_Train)
    rmse_train = mean_squared_error(y_train, pred_train) ** 0.5
    r2_train = r2_score(y_train, pred_train)

    predicted_cv = cross_val_predict(model, X_Train, y_train, cv=10)
    rmse_cv = mean_squared_error(y_train, predicted_cv) ** 0.5
    r2_cv = r2_score(y_train, predicted_cv)

    predicted_test = model.predict(X_test)
    rmse_test = mean_squared_error(y_test, predicted_test) ** 0.5
    r2_test = r2_score(y_test, predicted_test)

    return {
        "Set_Index": set_index,
        "Features": str(features),
        "Best_Params": str(best_params),
        "Model": model,
        "RMSE_Train": rmse_train,
        "R2_Train": r2_train,
        "RMSE_CV": rmse_cv,
        "R2_CV": r2_cv,
        "RMSE_Test": rmse_test,
        "R2_Test": r2_test
    }

# --- Main Execution Loop ---
combined_metrics = []
base_path = "/content/drive/MyDrive/datasets_Rayeed_GSV_POI_LU/"

for i, features in enumerate(feature_sets, 1):
    print(f"\n{'='*60}")
    print(f"Processing Feature Set {i}: {features}")
    print(f"{'='*60}")

    try:
        result = evaluate_ridge_model(features, i)

        # Add metrics to the combined list (excluding the model object)
        metric_record = {k: v for k, v in result.items() if k != 'Model'}
        combined_metrics.append(metric_record)

        print(f"Set {i} Best Params: {result['Best_Params']}")
        print(f"Set {i} Test RMSE: {result['RMSE_Test']:.4f}")

        # --- Generate and Save Predictions ---
        model = result['Model']
        x_pred_st_subset = x_pred_st[features]
        predictions = model.predict(x_pred_st_subset)

        # 1. Save Predictions
        pred_filename = f"predictions_dry_model_set_{i}_ridge.csv"
        predictions_df = pd.DataFrame(predictions, index=x_pred_st.index, columns=["Predictions"])
        predictions_df.to_csv(base_path + pred_filename)

        # 2. Save Prediction Stats
        stats_filename = f"predictions_dry_model_set_{i}_ridge_stats.csv"
        stats = predictions_df.describe()
        stats.to_csv(base_path + stats_filename)

        print(f"--> Saved predictions for Set {i}")

    except ValueError as e:
        if "Input contains NaN" in str(e):
            print(f"Skipping Set {i} due to NaN values in features.")
        else:
            raise e

# --- Save Combined Performance Metrics ---
if combined_metrics:
    print(f"\n{'='*60}")
    print("Saving Combined Performance Metrics...")

    metrics_df = pd.DataFrame(combined_metrics)

    # Save to a single CSV file
    combined_filename = "dry_model_ridge_combined_performance_metrics.csv"
    metrics_df.to_csv(base_path + combined_filename, index=False)

    print(f"Successfully saved combined metrics to: {base_path + combined_filename}")
    print(metrics_df)
else:
    print("No metrics were generated.")

[I 2025-12-21 21:09:31,244] A new study created in memory with name: Ridge_RMSE_Set_1
[I 2025-12-21 21:09:31,290] Trial 0 finished with value: 37.61676730788584 and parameters: {'alpha': 992.8263581797665}. Best is trial 0 with value: 37.61676730788584.
[I 2025-12-21 21:09:31,336] Trial 1 finished with value: 37.70249760178023 and parameters: {'alpha': 1169.609084268032}. Best is trial 0 with value: 37.61676730788584.
[I 2025-12-21 21:09:31,381] Trial 2 finished with value: 26.351400709882807 and parameters: {'alpha': 15.565926008120556}. Best is trial 2 with value: 26.351400709882807.



Processing Feature Set 1: ['gsv_path_p1500', 'gsv_sidewalk_p500', 'gsv_fence_p1500', 'gsv_wall_p250']


[I 2025-12-21 21:09:31,428] Trial 3 finished with value: 38.127899300159896 and parameters: {'alpha': 8298.812703605829}. Best is trial 2 with value: 26.351400709882807.
[I 2025-12-21 21:09:31,484] Trial 4 finished with value: 38.06224505808091 and parameters: {'alpha': 4317.0510625099805}. Best is trial 2 with value: 26.351400709882807.
[I 2025-12-21 21:09:31,527] Trial 5 finished with value: 23.5225809900346 and parameters: {'alpha': 1.289320506448353}. Best is trial 5 with value: 23.5225809900346.
[I 2025-12-21 21:09:31,573] Trial 6 finished with value: 23.805333541816193 and parameters: {'alpha': 3.5228917453597255}. Best is trial 5 with value: 23.5225809900346.
[I 2025-12-21 21:09:31,618] Trial 7 finished with value: 24.54933869782083 and parameters: {'alpha': 7.2849761930129455}. Best is trial 5 with value: 23.5225809900346.
[I 2025-12-21 21:09:31,665] Trial 8 finished with value: 23.639523661607047 and parameters: {'alpha': 2.405112601563341}. Best is trial 5 with value: 23.5225

Set 1 Best Params: {'alpha': 0.15402560326172424}
Set 1 Test RMSE: 21.5019
--> Saved predictions for Set 1

Processing Feature Set 2: ['gsv_path_p1500', 'gsv_sidewalk_p500', 'gsv_fence_p1500', 'land_Ar_MC_p_750']


[I 2025-12-21 21:09:36,362] Trial 2 finished with value: 25.82474693048187 and parameters: {'alpha': 0.02642310617825336}. Best is trial 2 with value: 25.82474693048187.
[I 2025-12-21 21:09:36,420] Trial 3 finished with value: 26.634564876589128 and parameters: {'alpha': 16.591177080283433}. Best is trial 2 with value: 25.82474693048187.
[I 2025-12-21 21:09:36,474] Trial 4 finished with value: 23.453233648458873 and parameters: {'alpha': 0.9127752641276536}. Best is trial 4 with value: 23.453233648458873.
[I 2025-12-21 21:09:36,531] Trial 5 finished with value: 26.68018080740266 and parameters: {'alpha': 16.82329926383687}. Best is trial 4 with value: 23.453233648458873.
[I 2025-12-21 21:09:36,587] Trial 6 finished with value: 36.33178962975407 and parameters: {'alpha': 287.976229410798}. Best is trial 4 with value: 23.453233648458873.
[I 2025-12-21 21:09:36,649] Trial 7 finished with value: 36.86231221337623 and parameters: {'alpha': 415.2313035117005}. Best is trial 4 with value: 23.

Set 2 Best Params: {'alpha': 0.8528359633654107}
Set 2 Test RMSE: 21.6259
--> Saved predictions for Set 2

Processing Feature Set 3: ['gsv_path_p1500', 'gsv_sidewalk_p500', 'gsv_fence_p1500', 'gsv_tree_p1000']


[I 2025-12-21 21:09:42,154] Trial 3 finished with value: 37.987058310330475 and parameters: {'alpha': 3925.5156189409695}. Best is trial 1 with value: 26.361397870253462.
[I 2025-12-21 21:09:42,195] Trial 4 finished with value: 30.02975661039564 and parameters: {'alpha': 56.46075151498099}. Best is trial 1 with value: 26.361397870253462.
[I 2025-12-21 21:09:42,245] Trial 5 finished with value: 25.167515627985946 and parameters: {'alpha': 0.33166537883851377}. Best is trial 5 with value: 25.167515627985946.
[I 2025-12-21 21:09:42,285] Trial 6 finished with value: 25.353543838685688 and parameters: {'alpha': 0.013139862363277955}. Best is trial 5 with value: 25.167515627985946.
[I 2025-12-21 21:09:42,330] Trial 7 finished with value: 24.867951807154075 and parameters: {'alpha': 1.0171087283874607}. Best is trial 7 with value: 24.867951807154075.
[I 2025-12-21 21:09:42,375] Trial 8 finished with value: 25.294878608617122 and parameters: {'alpha': 0.10710557720403349}. Best is trial 7 with

Set 3 Best Params: {'alpha': 4.737205860137392}
Set 3 Test RMSE: 21.8888
--> Saved predictions for Set 3

Processing Feature Set 4: ['gsv_path_p1500', 'gsv_sidewalk_p500', 'gsv_house_p500', 'gsv_fence_p1500']


[I 2025-12-21 21:09:47,148] Trial 4 finished with value: 38.04018199084196 and parameters: {'alpha': 5095.643753650747}. Best is trial 3 with value: 23.406115810582598.
[I 2025-12-21 21:09:47,195] Trial 5 finished with value: 34.67529321081424 and parameters: {'alpha': 186.20647153660585}. Best is trial 3 with value: 23.406115810582598.
[I 2025-12-21 21:09:47,240] Trial 6 finished with value: 38.04450241757612 and parameters: {'alpha': 5238.853844604563}. Best is trial 3 with value: 23.406115810582598.
[I 2025-12-21 21:09:47,299] Trial 7 finished with value: 23.406815409775767 and parameters: {'alpha': 0.0352195057172712}. Best is trial 3 with value: 23.406115810582598.
[I 2025-12-21 21:09:47,356] Trial 8 finished with value: 34.74170102723852 and parameters: {'alpha': 190.6862860103411}. Best is trial 3 with value: 23.406115810582598.
[I 2025-12-21 21:09:47,396] Trial 9 finished with value: 23.406724857560732 and parameters: {'alpha': 0.03755556656770351}. Best is trial 3 with value: 

Set 4 Best Params: {'alpha': 0.45541848165611454}
Set 4 Test RMSE: 22.9650
--> Saved predictions for Set 4

Saving Combined Performance Metrics...
Successfully saved combined metrics to: /content/drive/MyDrive/datasets_Rayeed_GSV_POI_LU/dry_model_ridge_combined_performance_metrics.csv
   Set_Index                                           Features  \
0          1  ['gsv_path_p1500', 'gsv_sidewalk_p500', 'gsv_f...   
1          2  ['gsv_path_p1500', 'gsv_sidewalk_p500', 'gsv_f...   
2          3  ['gsv_path_p1500', 'gsv_sidewalk_p500', 'gsv_f...   
3          4  ['gsv_path_p1500', 'gsv_sidewalk_p500', 'gsv_h...   

                      Best_Params  RMSE_Train  R2_Train    RMSE_CV     R2_CV  \
0  {'alpha': 0.15402560326172424}   20.198748  0.684748  23.476069  0.574147   
1   {'alpha': 0.8528359633654107}   20.120249  0.687193  23.451847  0.575025   
2    {'alpha': 4.737205860137392}   19.960257  0.692148  24.366602  0.541225   
3  {'alpha': 0.45541848165611454}   20.171479  0.685598  2

lasso

In [ ]:
import pandas as pd
import optuna
from sklearn.linear_model import Lasso
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import cross_val_predict

# Define the list of feature sets to iterate through
feature_sets = [
    ['gsv_path_p1500', 'gsv_sidewalk_p500', 'gsv_fence_p1500', 'gsv_wall_p250'],
    ['gsv_path_p1500', 'gsv_sidewalk_p500', 'gsv_fence_p1500', 'land_Ar_MC_p_750'],
    ['gsv_path_p1500', 'gsv_sidewalk_p500', 'gsv_fence_p1500', 'gsv_tree_p1000'],
    ['gsv_path_p1500', 'gsv_sidewalk_p500', 'gsv_house_p500', 'gsv_fence_p1500']
]

def evaluate_lasso_model(features, set_index):
    X_Train = X_st[features]
    X_test = X_test_st[features]

    def objective(trial):
        # Optimization parameter for Lasso: Alpha
        alpha = trial.suggest_float('alpha', 0.0001, 100.0, log=True)

        model = Lasso(alpha=alpha, random_state=42)

        predicted = cross_val_predict(model, X_Train, y_train, cv=10)
        return mean_squared_error(y_train, predicted) ** 0.5

    # Unique study name
    study_name = f"Lasso_RMSE_Set_{set_index}"
    study = optuna.create_study(direction='minimize', study_name=study_name)
    study.optimize(objective, n_trials=100)

    best_params = study.best_params

    # Rebuild final model
    model = Lasso(
        alpha=best_params['alpha'],
        random_state=42
    )

    model.fit(X_Train, y_train)

    # --- Calculate True RMSE ---
    pred_train = model.predict(X_Train)
    rmse_train = mean_squared_error(y_train, pred_train) ** 0.5
    r2_train = r2_score(y_train, pred_train)

    predicted_cv = cross_val_predict(model, X_Train, y_train, cv=10)
    rmse_cv = mean_squared_error(y_train, predicted_cv) ** 0.5
    r2_cv = r2_score(y_train, predicted_cv)

    predicted_test = model.predict(X_test)
    rmse_test = mean_squared_error(y_test, predicted_test) ** 0.5
    r2_test = r2_score(y_test, predicted_test)

    return {
        "Set_Index": set_index,
        "Features": str(features),
        "Best_Params": str(best_params),
        "Model": model,
        "RMSE_Train": rmse_train,
        "R2_Train": r2_train,
        "RMSE_CV": rmse_cv,
        "R2_CV": r2_cv,
        "RMSE_Test": rmse_test,
        "R2_Test": r2_test
    }

# --- Main Execution Loop ---
combined_metrics = []
base_path = "/content/drive/MyDrive/datasets_Rayeed_GSV_POI_LU/"

for i, features in enumerate(feature_sets, 1):
    print(f"\n{'='*60}")
    print(f"Processing Feature Set {i}: {features}")
    print(f"{'='*60}")

    try:
        result = evaluate_lasso_model(features, i)

        # Add metrics to the combined list (excluding the model object)
        metric_record = {k: v for k, v in result.items() if k != 'Model'}
        combined_metrics.append(metric_record)

        print(f"Set {i} Best Params: {result['Best_Params']}")
        print(f"Set {i} Test RMSE: {result['RMSE_Test']:.4f}")

        # --- Generate and Save Predictions ---
        model = result['Model']
        x_pred_st_subset = x_pred_st[features]
        predictions = model.predict(x_pred_st_subset)

        # 1. Save Predictions
        pred_filename = f"predictions_dry_model_set_{i}_lasso.csv"
        predictions_df = pd.DataFrame(predictions, index=x_pred_st.index, columns=["Predictions"])
        predictions_df.to_csv(base_path + pred_filename)

        # 2. Save Prediction Stats
        stats_filename = f"predictions_dry_model_set_{i}_lasso_stats.csv"
        stats = predictions_df.describe()
        stats.to_csv(base_path + stats_filename)

        print(f"--> Saved predictions for Set {i}")

    except ValueError as e:
        if "Input contains NaN" in str(e):
            print(f"Skipping Set {i} due to NaN values in features.")
        else:
            raise e

# --- Save Combined Performance Metrics ---
if combined_metrics:
    print(f"\n{'='*60}")
    print("Saving Combined Performance Metrics...")

    metrics_df = pd.DataFrame(combined_metrics)

    # Save to a single CSV file
    combined_filename = "dry_model_lasso_combined_performance_metrics.csv"
    metrics_df.to_csv(base_path + combined_filename, index=False)

    print(f"Successfully saved combined metrics to: {base_path + combined_filename}")
    print(metrics_df)
else:
    print("No metrics were generated.")

[I 2025-12-21 21:09:52,456] A new study created in memory with name: Lasso_RMSE_Set_1
[I 2025-12-21 21:09:52,524] Trial 0 finished with value: 23.529799253521805 and parameters: {'alpha': 0.1390935526820363}. Best is trial 0 with value: 23.529799253521805.



Processing Feature Set 1: ['gsv_path_p1500', 'gsv_sidewalk_p500', 'gsv_fence_p1500', 'gsv_wall_p250']


[I 2025-12-21 21:09:52,590] Trial 1 finished with value: 23.478477688080186 and parameters: {'alpha': 0.003800990686450965}. Best is trial 1 with value: 23.478477688080186.
[I 2025-12-21 21:09:52,668] Trial 2 finished with value: 23.47713006987289 and parameters: {'alpha': 0.00019205028911381541}. Best is trial 2 with value: 23.47713006987289.
[I 2025-12-21 21:09:52,736] Trial 3 finished with value: 24.599125991921028 and parameters: {'alpha': 2.5466832801146144}. Best is trial 2 with value: 23.47713006987289.
[I 2025-12-21 21:09:52,812] Trial 4 finished with value: 23.477366131362885 and parameters: {'alpha': 0.0008144079522386156}. Best is trial 2 with value: 23.47713006987289.
[I 2025-12-21 21:09:52,885] Trial 5 finished with value: 23.477521344162323 and parameters: {'alpha': 0.001245390959258134}. Best is trial 2 with value: 23.47713006987289.
[I 2025-12-21 21:09:52,958] Trial 6 finished with value: 23.477103699091863 and parameters: {'alpha': 0.0001317898526687626}. Best is trial

Set 1 Best Params: {'alpha': 0.00010042036259733778}
Set 1 Test RMSE: 21.5061
--> Saved predictions for Set 1

Processing Feature Set 2: ['gsv_path_p1500', 'gsv_sidewalk_p500', 'gsv_fence_p1500', 'land_Ar_MC_p_750']


[I 2025-12-21 21:09:58,228] Trial 2 finished with value: 26.307771883700543 and parameters: {'alpha': 0.0008999974181728051}. Best is trial 2 with value: 26.307771883700543.
[I 2025-12-21 21:09:58,273] Trial 3 finished with value: 28.703406172992917 and parameters: {'alpha': 8.055523575702509}. Best is trial 2 with value: 26.307771883700543.
[I 2025-12-21 21:09:58,323] Trial 4 finished with value: 38.20023221905141 and parameters: {'alpha': 26.061714553247732}. Best is trial 2 with value: 26.307771883700543.
[I 2025-12-21 21:09:58,372] Trial 5 finished with value: 23.684492606022825 and parameters: {'alpha': 0.11218538035267604}. Best is trial 5 with value: 23.684492606022825.
[I 2025-12-21 21:09:58,422] Trial 6 finished with value: 26.108106188392053 and parameters: {'alpha': 0.006434357253150537}. Best is trial 5 with value: 23.684492606022825.
[I 2025-12-21 21:09:58,470] Trial 7 finished with value: 24.000663085622687 and parameters: {'alpha': 1.0423036118478524}. Best is trial 5 wi

Set 2 Best Params: {'alpha': 0.1432979263472758}
Set 2 Test RMSE: 21.7065
--> Saved predictions for Set 2

Processing Feature Set 3: ['gsv_path_p1500', 'gsv_sidewalk_p500', 'gsv_fence_p1500', 'gsv_tree_p1000']


[I 2025-12-21 21:10:03,543] Trial 3 finished with value: 26.039223700972403 and parameters: {'alpha': 1.5377516424160607}. Best is trial 0 with value: 25.36215308765703.
[I 2025-12-21 21:10:03,598] Trial 4 finished with value: 38.199612806744504 and parameters: {'alpha': 71.37843479873507}. Best is trial 0 with value: 25.36215308765703.
[I 2025-12-21 21:10:03,653] Trial 5 finished with value: 25.36213515693254 and parameters: {'alpha': 0.00045048866108847257}. Best is trial 5 with value: 25.36213515693254.
[I 2025-12-21 21:10:03,702] Trial 6 finished with value: 25.386505017058873 and parameters: {'alpha': 0.08455761450604797}. Best is trial 5 with value: 25.36213515693254.
[I 2025-12-21 21:10:03,749] Trial 7 finished with value: 25.37702990473151 and parameters: {'alpha': 0.052845528458390835}. Best is trial 5 with value: 25.36213515693254.
[I 2025-12-21 21:10:03,794] Trial 8 finished with value: 25.36204969228038 and parameters: {'alpha': 0.0001078166573271331}. Best is trial 8 with 

Set 3 Best Params: {'alpha': 0.00010010584505439082}
Set 3 Test RMSE: 21.9038
--> Saved predictions for Set 3

Processing Feature Set 4: ['gsv_path_p1500', 'gsv_sidewalk_p500', 'gsv_house_p500', 'gsv_fence_p1500']


[I 2025-12-21 21:10:10,077] Trial 3 finished with value: 23.424055060731586 and parameters: {'alpha': 0.07083457814865317}. Best is trial 0 with value: 23.40851655715659.
[I 2025-12-21 21:10:10,128] Trial 4 finished with value: 23.435661975401306 and parameters: {'alpha': 0.12046681604005234}. Best is trial 0 with value: 23.40851655715659.
[I 2025-12-21 21:10:10,181] Trial 5 finished with value: 23.408661883893984 and parameters: {'alpha': 0.0018044708091323487}. Best is trial 0 with value: 23.40851655715659.
[I 2025-12-21 21:10:10,228] Trial 6 finished with value: 38.199612806744504 and parameters: {'alpha': 56.25593901011403}. Best is trial 0 with value: 23.40851655715659.
[I 2025-12-21 21:10:10,279] Trial 7 finished with value: 23.673881289508824 and parameters: {'alpha': 0.8958331151757499}. Best is trial 0 with value: 23.40851655715659.
[I 2025-12-21 21:10:10,329] Trial 8 finished with value: 23.416937152174363 and parameters: {'alpha': 0.039419932475963246}. Best is trial 0 with 

Set 4 Best Params: {'alpha': 0.00015999226721467264}
Set 4 Test RMSE: 22.8563
--> Saved predictions for Set 4

Saving Combined Performance Metrics...
Successfully saved combined metrics to: /content/drive/MyDrive/datasets_Rayeed_GSV_POI_LU/dry_model_lasso_combined_performance_metrics.csv
   Set_Index                                           Features  \
0          1  ['gsv_path_p1500', 'gsv_sidewalk_p500', 'gsv_f...   
1          2  ['gsv_path_p1500', 'gsv_sidewalk_p500', 'gsv_f...   
2          3  ['gsv_path_p1500', 'gsv_sidewalk_p500', 'gsv_f...   
3          4  ['gsv_path_p1500', 'gsv_sidewalk_p500', 'gsv_h...   

                         Best_Params  RMSE_Train  R2_Train    RMSE_CV  \
0  {'alpha': 0.00010042036259733778}   20.198238  0.684764  23.477092   
1      {'alpha': 0.1432979263472758}   20.105568  0.687650  23.539937   
2  {'alpha': 0.00010010584505439082}   19.711217  0.699782  25.362047   
3  {'alpha': 0.00015999226721467264}   20.166758  0.685745  23.408236   

      R2_